# Spur Review: three-extension experimental POC

Prove the data plane before building an app. This notebook uses the saved **@github-pat** connection for a small GitHub inbox, then combines **DuckTails**, the rebuilt **Spur Graph** extension and **DuckPGQ** in DuckDB 2.0.

Experiments: runtime and provider preflight; live GitHub discovery/PR pinning; exact Git source extraction; base-revision history/blame; typed property-graph queries; component and stakeholder joins; a cited evidence packet for later AI review.

The initial case is DuckDB PR #28731. Measured native facts, curated cross-file relationships and historical co-change remain distinct. This POC performs read-only GitHub and Git queries. It does not execute an AI review or publish a GitHub review.


In [2]:
import duckdb, json, time, hashlib, subprocess
from pathlib import Path
from datetime import datetime, timezone
assert duckdb.__version__ == '2.0.0', duckdb.__version__
ARTIFACT = Path('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-17-review-extensions-poc')
ARTIFACT.mkdir(parents=True, exist_ok=True)
EXTENSIONS = Path('/Users/kevintruong/project/SpurCode/.spur/artifacts/review-extensions-poc/extensions')
EXTENSIONS.mkdir(parents=True, exist_ok=True)
SPUR_BINARY = Path('/Users/kevintruong/project/SpurCode/.spur/artifacts/explore-audit-bd-2bhj-native/spur_graph.duckdb_extension')
def rows(sql, params=None):
    result = duckdb.execute(sql, params or [])
    return [dict(zip([c[0] for c in result.description], row)) for row in result.fetchall()]
def save(name, value):
    (ARTIFACT/(name+'.json')).write_text(json.dumps(value,default=str,indent=2))
duckdb.execute("SET extension_directory = '"+str(EXTENSIONS)+"'")
install_receipts = []
for statement in ["INSTALL duck_tails FROM community",
                  "INSTALL '"+str(SPUR_BINARY)+"'",
                  "INSTALL duckpgq FROM community",
                  "LOAD duck_tails","LOAD spur_graph","LOAD duckpgq"]:
    started = time.monotonic()
    duckdb.execute(statement)
    install_receipts.append({'statement':statement,'elapsed_seconds':round(time.monotonic()-started,3),'status':'ok'})
# The grammar setting must be a separate statement after LOAD.
duckdb.execute("SET active_grammar_extensions = ['duckpgq']")
extension_inventory = rows("""
    SELECT extension_name,extension_version,installed,loaded,install_mode,installed_from
    FROM duckdb_extensions() WHERE extension_name IN ('duck_tails','spur_graph','duckpgq')
    ORDER BY extension_name
""")
assert len(extension_inventory)==3 and all(r['loaded'] for r in extension_inventory)
save('extensions',{'runtime':duckdb.__version__,'inventory':extension_inventory,'receipts':install_receipts,
                   'spur_graph_info':rows('SELECT spur_graph_info() AS metadata'),
                   'captured_at_utc':datetime.now(timezone.utc).isoformat()})
print(json.dumps(extension_inventory,indent=2))
print('All three extensions installed and loaded in one DuckDB 2.0 connection.')


[
  {
    "extension_name": "duck_tails",
    "extension_version": "v1.9.0",
    "installed": true,
    "loaded": true,
    "install_mode": "REPOSITORY",
    "installed_from": "community"
  },
  {
    "extension_name": "duckpgq",
    "extension_version": "9c11bc192a",
    "installed": true,
    "loaded": true,
    "install_mode": "REPOSITORY",
    "installed_from": "community"
  },
  {
    "extension_name": "spur_graph",
    "extension_version": "0.1.0",
    "installed": true,
    "loaded": true,
    "install_mode": "CUSTOM_PATH",
    "installed_from": "/Users/kevintruong/project/SpurCode/.spur/artifacts/explore-audit-bd-2bhj-native/spur_graph.duckdb_extension"
  }
]
All three extensions installed and loaded in one DuckDB 2.0 connection.


In [3]:
-- Minimal DuckPGQ smoke test. This is a synthetic grammar/engine fixture.
CREATE OR REPLACE TABLE pgq_probe_nodes (id BIGINT PRIMARY KEY, name VARCHAR);
INSERT INTO pgq_probe_nodes VALUES (1,'changed'),(2,'caller'),(3,'consumer');
CREATE OR REPLACE TABLE pgq_probe_edges (src BIGINT,dst BIGINT);
INSERT INTO pgq_probe_edges VALUES (1,2),(2,3);
CREATE OR REPLACE PROPERTY GRAPH pgq_probe
  VERTEX TABLES (pgq_probe_nodes LABEL symbol)
  EDGE TABLES (pgq_probe_edges
    SOURCE KEY (src) REFERENCES pgq_probe_nodes (id)
    DESTINATION KEY (dst) REFERENCES pgq_probe_nodes (id)
    LABEL relates);
SELECT * FROM GRAPH_TABLE(pgq_probe
  MATCH p = ANY SHORTEST (a:symbol)-[e:relates]->{1,3}(b:symbol)
  WHERE a.id = 1
  COLUMNS (a.name AS source_name,b.name AS target_name,path_length(p) AS hops))
ORDER BY hops,target_name;

source_name,target_name,hops
changed,caller,1
changed,consumer,2


In [4]:
# SPUR datasource setup cell v1
# This cell is managed by SPUR and runs when connections change.
# SPUR DuckDB bootstrap
import duckdb
import sys
_SPUR_DUCKDB_EXTENSION_PATH = "/Users/kevintruong/.spur/extensions/spur_rest.duckdb_extension"
# Shared by notebook setup and classic SQL cells. Preserve the generic path for
# older bundles; new bundles carry exact native ABI artifacts per engine.
from pathlib import Path as _SpurExtensionPath

_spur_extension = _SpurExtensionPath(_SPUR_DUCKDB_EXTENSION_PATH)
_spur_versioned_extension = (
    _spur_extension.parent
    / ("duckdb-v" + duckdb.__version__)
    / _spur_extension.name
)
if _spur_versioned_extension.is_file():
    _SPUR_DUCKDB_EXTENSION_PATH = str(_spur_versioned_extension)

_SPUR_DUCKDB_GATEWAY_COMPATIBLE = duckdb.__version__ in ("1.4.4", "1.5.5")

if duckdb.__version__ == "2.0.0":
    from platform import machine as _spur_machine

    # Never leave the generic legacy path available to an exact-2.0 caller,
    # including explicit datasource setup when the artifact is missing.
    _SPUR_DUCKDB_EXTENSION_PATH = str(_spur_versioned_extension)
    _spur_gateway_error = ""
    if sys.platform != "darwin" or _spur_machine().lower() not in ("arm64", "aarch64"):
        _spur_gateway_error = "the 2.0 gateway is qualified only on macOS ARM64"
    else:
        try:
            # DuckDB's footer is eight 32-byte fields and a 256-byte signature,
            # preceded by its 22-byte metadata marker. Check final signed bytes.
            with _spur_versioned_extension.open("rb") as _spur_artifact:
                _spur_artifact.seek(-534, 2)
                _spur_footer = _spur_artifact.read(534)
            _spur_fields = [
                _spur_footer[22 + i * 32:22 + (i + 1) * 32].rstrip(b"\x00")
                for i in range(8)
            ]
            if (
                _spur_footer[:22] != b"\x00\x93\x04\x10duckdb_signature\x80\x04"
                or _spur_fields[3] != b"C_STRUCT_UNSTABLE"
                or _spur_fields[5:] != [b"v2.0.0", b"osx_arm64", b"4"]
            ):
                _spur_gateway_error = "the artifact footer does not match DuckDB 2.0.0 on macOS ARM64"
        except OSError:
            _spur_gateway_error = "the exact 2.0.0 artifact is missing or unreadable"
    _SPUR_DUCKDB_GATEWAY_COMPATIBLE = not _spur_gateway_error
    if _spur_gateway_error:
        print(
            f"SPUR: {_spur_gateway_error}. Install or rebuild Jute with the matching "
            f"gateway artifact ({_SPUR_DUCKDB_EXTENSION_PATH}), then restart the notebook "
            "kernel. Pure SQL remains available.",
            file=sys.stderr,
        )

if (
    sys.platform in ("linux", "darwin")
    and _SPUR_DUCKDB_GATEWAY_COMPATIBLE
    and "_SPUR_DUCKDB_HOST_SYMBOLS" not in globals()
):
    import ctypes as _spur_ctypes
    import _duckdb as _spur_host_duckdb

    # Promote the already-loaded Python engine's symbols. This opens the same
    # module handle; it neither imports another engine nor replaces connections.
    _SPUR_DUCKDB_HOST_SYMBOLS = _spur_ctypes.CDLL(
        _spur_host_duckdb.__file__, mode=_spur_ctypes.RTLD_GLOBAL
    )

_SPUR_DUCKDB_EXTENSION_SQL = _SPUR_DUCKDB_EXTENSION_PATH.replace("'", "''")

if "_SPUR_DUCKDB_CONNECTION" not in globals():
    _SPUR_DUCKDB_CONNECTION = duckdb.connect(
        database=":memory:",
        config={"allow_unsigned_extensions": "true",
                 "autoload_known_extensions": "false",
                 "autoinstall_known_extensions": "false"},
    )

duckdb.set_default_connection(_SPUR_DUCKDB_CONNECTION)
# Embedded helper source contains no token, credential path or capability lease.
# The approved local launcher passes a private inherited socket descriptor, not
# executable source or a notebook/widget message. Same-kernel code is trusted.
def _spur_managed_connection(db):
    """Only the host's selected connection is subject to managed policy."""
    import types
    connection = db.default_connection() if isinstance(db, types.ModuleType) else db
    if connection is not globals().get("_SPUR_DUCKDB_CONNECTION"):
        return None
    try:
        connection.sql("SET autoload_known_extensions = false")
        connection.sql("SET autoinstall_known_extensions = false")
    except Exception:
        return None  # existing locked/custom connections have an explicit boundary
    return connection


# Required SpurLab baseline: JSON/parquet are statically linked in the managed
# wheel; httpfs supplies remote files and REST gateway dependencies; spur_rest is
# the separately packaged, host-verified gateway (never a repository download).
# Remote datasource attach code explicitly installs/loads its provider adapter.
# Spatial and Jev are optional. No browsing/query path calls operate("install").
# Fixed metadata query only. Identifiers and descriptions remain inert JSON; no
# catalog value is interpolated as SQL. The limits bound the shell reply too.
def _spur_function_catalog(db):
    import json
    limit = 10000
    rows = db.sql("SELECT database_name, schema_name, function_name, function_type, "
                  "parameters, parameter_types, varargs, return_type, description, examples, function_oid "
                  "FROM duckdb_functions() ORDER BY database_name, schema_name, function_name, "
                  "function_type, parameter_types, varargs LIMIT 10001").fetchall()
    functions, size = [], 0
    complete = len(rows) <= limit
    for row in rows[:limit]:
        value = dict(zip(("database_name", "schema_name", "name", "function_type", "parameters",
                          "parameter_types", "varargs", "return_type", "description", "examples", "function_oid"), row))
        value["ownership"] = "unknown"
        encoded = json.dumps(value, ensure_ascii=True)
        # Oversized metadata is omitted, with incomplete coverage made explicit.
        if len(encoded) > 65536 or size + len(encoded) > 4 * 1024 * 1024:
            complete = False
            continue
        functions.append(value)
        size += len(encoded)
    return {"status": "complete" if complete else "incomplete", "functions": functions,
            "truncated": not complete}


def _spur_function_identity(value):
    return (value["database_name"], value["schema_name"], value["name"],
            value["function_type"], tuple(value["parameter_types"]), value["varargs"])


def _spur_snapshot_function_registration(db):
    try:
        snapshot = _spur_function_catalog(db)
        if snapshot["status"] == "complete" and snapshot.get("truncated") is False:
            return snapshot
    except Exception:
        pass  # Optional metadata must not prevent an authorized extension load.
    return None


def _spur_record_function_ownership(db, extension, before):
    """Retain registrations observed around a verified load on this connection."""
    if before is None:
        return
    try:
        if before["status"] != "complete" or before.get("truncated") is not False:
            return
        after = _spur_snapshot_function_registration(db)
        if after is None:
            return
        def key(row):
            return (*_spur_function_identity(row), row.get("return_type"), row["function_oid"])
        previous = {key(row) for row in before["functions"]}
        registrations = globals().setdefault("_SPUR_DUCKDB_FUNCTION_OWNERSHIP", {})
        retained = registrations.get(id(db))
        owners = dict(retained[1]) if retained is not None and retained[0] is db else {}
        additions = [key(row) for row in after["functions"]
                     if row["function_oid"] is not None and key(row) not in previous]
        for identity in additions:
            owners[identity] = set(owners.get(identity, ())) | {extension}
        registrations[id(db)] = (db, owners)
    except Exception:
        pass  # Incomplete/failed evidence stays unattributed.


class _SpurDuckDBRuntime:
    BASELINE = ("json", "parquet", "httpfs", "spur_rest")
    REPOSITORIES = {
        # DuckDB's built-in installer handles HTTP without httpfs; HTTPS itself
        # requires httpfs. Use the official signed distribution bootstrap, never
        # temporarily re-enable autoload to download its own dependency.
        "httpfs": "http://extensions.duckdb.org",
        "spatial": "http://extensions.duckdb.org",
        "jev": "http://community-extensions.duckdb.org",
    }

    def __init__(self, db, platform):
        self.db, self.platform = db, platform
        self.entries = {}
        self.inventories = {}  # inert metadata; never activation authority
        self.verified = {}  # at most one private verified copy per curated entry
        self.loaded = set()  # artifact verification is not selected-connection load

    def _entry(self, name):
        return self.entries.setdefault(name, {
            "name": name, "installed": None,
            "desired_enabled": name in self.BASELINE, "loaded": None,
            "provider_health": "unknown", "restart_required": False,
            "activation_transport_restart": False,
            "stage": "idle", "reason": None, "policy": "unknown",
            "artifact_digest": None, "signature_verified": False,
        })

    def _observe(self, db, name):
        rows = db.sql(
            "SELECT loaded, installed, install_path, installed_from, install_mode "
            "FROM duckdb_extensions() WHERE extension_name = ?", params=[name]
        ).fetchall()
        return rows[0] if rows else None

    def _refresh(self, entry):
        row = self._observe(self.db, entry["name"])
        entry["loaded"] = bool(row and row[0])
        entry["installed"] = bool(row and row[1])
        return row

    def _artifact(self, name, install, cancelled):
        # Verification happens in a separate database with signature and ABI
        # checks ON, even when the selected gateway connection allows unsigned
        # bundled code. A repository label or hash alone is not signing evidence.
        import duckdb
        import hashlib
        from pathlib import Path
        import shutil
        import tempfile
        version = self.db.sql("SELECT version()").fetchall()[0][0]
        config = {"allow_unsigned_extensions": "false",
                  "allow_community_extensions": "true" if name == "jev" else "false",
                  "allow_extensions_metadata_mismatch": "false",
                  "autoload_known_extensions": "false",
                  "autoinstall_known_extensions": "false"}
        if version == "v2.0.0":
            directories = self.db.sql("SELECT current_setting('extension_directories')").fetchall()[0][0]
            if directories:
                config["extension_directories"] = directories
        else:
            directory = self.db.sql("SELECT current_setting('extension_directory')").fetchall()[0][0]
            config["home_directory"] = self.db.sql("SELECT current_setting('home_directory')").fetchall()[0][0]
            if directory:
                config["extension_directory"] = directory
        entry = self._entry(name)
        entry["signature_verified"] = False
        with duckdb.connect(config=config) as verifier:
            if (version not in ("v1.5.5", "v2.0.0")
                    or verifier.sql("SELECT version()").fetchall()[0][0] != version
                    or verifier.sql("PRAGMA platform").fetchall()[0][0] != self.platform):
                raise ValueError("runtime_mismatch")
            entry["stage"] = "installing" if install else "verifying"
            row = self._observe(verifier, name)
            if install and (row is None or not row[1]):
                # Install only when absent. DuckDB rejects INSTALL across HTTP/
                # HTTPS origin spellings even for the same official repository.
                # Cached artifacts follow the same origin/signature/ABI checks
                # below; never FORCE/UPDATE or silently replace rejected bytes.
                verifier.install_extension(name, repository_url=self.REPOSITORIES[name])
                row = self._observe(verifier, name)
            if row is None or not row[1]:
                raise ValueError("not_installed")
            entry["installed"] = True
            if cancelled():
                raise InterruptedError()
            entry["stage"] = "verifying"
            repository = self.REPOSITORIES[name]
            aliases = (repository, repository.replace("http:", "https:"),
                       "community" if name == "jev" else "core")
            if row[3] not in aliases or row[4] != "REPOSITORY":
                raise ValueError("repository_mismatch")
            # Freeze the bytes in a private directory before the verifier loads
            # them. Load those same bytes into the selected connection, never a
            # mutable cache alias. Same-user/kernel arbitrary code is trusted.
            private = tempfile.TemporaryDirectory(prefix="spur-duckdb-verified-")
            try:
                path = Path(private.name) / (name + ".duckdb_extension")
                shutil.copyfile(row[2], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).hexdigest()
                try:
                    before = _spur_function_catalog(verifier)
                except Exception:
                    before = None
                verifier.load_extension(str(path))  # engine checks signature/ABI
                if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
                    raise ValueError("artifact_changed")
                try:
                    after = _spur_function_catalog(verifier)
                    if before is None:
                        raise ValueError("inventory_failed")
                    previous_rows = {_spur_function_identity(f): f for f in before["functions"]}
                    functions = [f for f in after["functions"]
                                 if previous_rows.get(_spur_function_identity(f)) != f]
                    complete = before["status"] == after["status"] == "complete"
                    inventory = {"status": "complete" if complete else "incomplete",
                                 "functions": functions, "truncated": not complete}
                except Exception:
                    inventory = {"status": "failed", "functions": [], "truncated": False}
                inventory.update(artifact_digest=digest, provenance="isolated_signed_verifier_load_delta")
                self.inventories[name] = inventory
            except BaseException:
                private.cleanup()
                raise
            previous = self.verified.get(name)
            if previous is not None:
                previous.cleanup()
            self.verified[name] = private
            return str(path), digest

    def operate(self, name, action, cancelled=lambda: False):
        """Explicit, nonsecret job; called only on the kernel shell queue.

        Cancellation is cooperative at stages. It never interrupts a user query;
        installed but not loaded artifacts remain available for an explicit retry.
        There is no UNLOAD in DuckDB: disable reports loaded + restart_required.
        """
        entry = self._entry(name)
        entry["stage"], entry["reason"] = "resolving", None
        if action == "disable" and name not in self.BASELINE:
            entry["desired_enabled"] = False
            if name == "jev":
                _spur_close_jev_channel()
        try:
            if name not in (*self.BASELINE, "spatial", "jev"):
                raise ValueError("unsupported_extension")
            if self.db is None:
                raise ValueError("no_connection")
            if self.db is not globals().get("_SPUR_DUCKDB_CONNECTION"):
                entry["policy"] = "unenforceable"
                raise ValueError("unmanaged_connection")
            if (self.db.sql("SELECT version()").fetchall()[0][0] not in ("v1.5.5", "v2.0.0")
                    or self.db.sql("PRAGMA platform").fetchall()[0][0] != self.platform):
                raise ValueError("runtime_mismatch")
            if _spur_managed_connection(self.db) is None:
                entry["policy"] = "unenforceable"
                raise ValueError("policy_unenforceable")
            entry["policy"] = "enforced"
            self._refresh(entry)
            if action == "disable":
                if name in self.BASELINE:
                    raise ValueError("required_baseline")
                entry["desired_enabled"] = False
                entry["restart_required"] = entry["loaded"] or name == "jev"
                entry["activation_transport_restart"] = name == "jev"
                entry["reason"] = "restart_required" if entry["restart_required"] else None
                entry["stage"] = "succeeded"
                return dict(entry)
            if cancelled():
                raise InterruptedError()
            if action == "invalidate_credential_channel" and name == "jev":
                _spur_close_jev_channel()
                entry["restart_required"] = True
                entry["activation_transport_restart"] = True
                entry["reason"] = "restart_required"
            elif action == "enable":
                entry["desired_enabled"] = True
                if name == "jev" and not self._channel_ready():
                    entry["restart_required"] = True
                    entry["activation_transport_restart"] = True
                    entry["reason"] = self._channel_reason()
            elif action in ("install", "revalidate", "load", "prepare_verification"):
                if action == "prepare_verification" and (name not in self.REPOSITORIES or not entry["installed"]
                        or not entry["signature_verified"] or not entry["artifact_digest"]):
                    raise ValueError("not_installed")
                if action == "load" and not entry["desired_enabled"]:
                    raise ValueError("setup_required_without_autoload")
                if name in ("json", "parquet"):
                    row = self._observe(self.db, name)
                    if not row or row[4] != "STATICALLY_LINKED" or not row[0]:
                        raise ValueError("baseline_unavailable")
                elif name == "spur_rest":
                    # Owned by the bundled artifact installer/bootstrap. Never
                    # weaken signature policy or download a substitute here.
                    if not entry["loaded"]:
                        raise ValueError("baseline_unavailable")
                else:
                    if entry["loaded"] and name == "httpfs" and name not in self.loaded:
                        # Connection setup may have loaded the same signed baseline
                        # first. Accept only its exact-connection frozen-byte proof.
                        from pathlib import Path
                        import hashlib
                        proof = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {}).get((id(self.db), name))
                        if proof and proof[0] is self.db and proof[1] == "core":
                            try:
                                digest = hashlib.sha256((Path(proof[2].name) / "httpfs.duckdb_extension").read_bytes()).digest()
                                if digest == proof[3]:
                                    self.verified[name] = proof[2]
                                    self.loaded.add(name)
                                    entry.update(signature_verified=True, artifact_digest=digest.hex(),
                                                 restart_required=False, activation_transport_restart=False)
                            except OSError:
                                pass
                    if entry["loaded"] and name not in self.loaded:
                        entry["signature_verified"] = False
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = False
                        raise ValueError("preexisting_load_unverified")
                    if entry["loaded"] and action in ("load", "prepare_verification"):
                        # Explicit retry may reacquire discovery in the same isolated
                        # signature verifier; it must not change selected enablement.
                        if action == "prepare_verification" and self.inventories.get(name, {}).get("status") != "complete":
                            loaded_digest = entry["artifact_digest"]
                            _, refreshed_digest = self._artifact(name, False, cancelled)
                            if refreshed_digest != loaded_digest:
                                entry["restart_required"] = True
                                entry["activation_transport_restart"] = False
                                raise ValueError("artifact_changed")
                            entry["signature_verified"] = True
                        if name == "jev" and action == "load" and not self._channel_ready():
                            entry["restart_required"] = True
                            entry["activation_transport_restart"] = True
                            entry["reason"] = self._channel_reason()
                        if entry["restart_required"] and entry["activation_transport_restart"]:
                            entry["reason"] = self._channel_reason()
                        entry["stage"] = "succeeded"
                        return dict(entry)
                    if entry["loaded"]:
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = False
                        raise ValueError("restart_required")
                    path, digest = self._artifact(name, action == "install", cancelled)
                    entry["installed"] = True
                    entry["artifact_digest"] = digest
                    entry["signature_verified"] = True
                    if cancelled():
                        raise InterruptedError()
                    if action in ("load", "prepare_verification") and not entry["loaded"]:
                        entry["stage"] = "loading"
                        before = _spur_snapshot_function_registration(self.db)
                        self.db.load_extension(path)
                        self._refresh(entry)
                        if not entry["loaded"]:
                            raise ValueError("load_failed")
                        self.loaded.add(name)
                        _spur_record_function_ownership(self.db, name, before)
                    if name == "jev" and action != "prepare_verification" and not self._channel_ready():
                        entry["restart_required"] = True
                        entry["activation_transport_restart"] = True
                        entry["reason"] = self._channel_reason()
            elif action != "inspect":
                raise ValueError("unsupported_operation")
            if entry["restart_required"] and entry["activation_transport_restart"]:
                entry["reason"] = self._channel_reason()
            entry["stage"] = "succeeded"
        except (KeyboardInterrupt, InterruptedError):
            entry["reason"] = "cancelled"
        except ValueError as error:
            # Only our fixed reasons are exposed; never native exception text.
            allowed = {"runtime_mismatch", "unsupported_extension", "unmanaged_connection",
                       "required_baseline", "setup_required_without_autoload", "not_installed",
                       "repository_mismatch", "artifact_changed", "baseline_unavailable",
                       "preexisting_load_unverified", "load_failed", "unsupported_operation", "no_connection",
                       "policy_unenforceable", "restart_required"}
            entry["reason"] = str(error) if str(error) in allowed else "verification_failed"
        except Exception:
            entry["reason"] = {"installing": "install_failed", "loading": "load_failed",
                               "verifying": "verification_failed"}.get(entry["stage"], "policy_failed")
        return dict(entry)

    @staticmethod
    def _channel_ready():
        import os
        if os.name != "posix":
            return False
        try:
            channel = _spur_jev_channel()
            return channel is not None and channel.available()
        except Exception:
            return False

    @staticmethod
    def _channel_reason():
        import os
        return "restart_required" if os.name == "posix" else "credential_transport_unavailable"

    def close(self):
        """Release verified temporary copies after closing this connection."""
        for directory in self.verified.values():
            directory.cleanup()
        self.verified.clear()


def _spur_duckdb_job(platform, name, action, cancel_path=None):
    """Entry point for typed host jobs; no caller SQL and no credential payload."""
    from pathlib import Path
    global _SPUR_DUCKDB_RUNTIME
    db = globals().get("_SPUR_DUCKDB_CONNECTION")
    current = globals().get("_SPUR_DUCKDB_RUNTIME")
    if current is None or current.db is not db or current.platform != platform:
        current = _SPUR_DUCKDB_RUNTIME = _SpurDuckDBRuntime(db, platform)
    return current.operate(name, action, lambda: bool(cancel_path and Path(cancel_path).exists()))


def _spur_duckdb_baseline():
    """Load approved cached baseline only; missing artifacts need explicit setup.

    Used by both root bootstrap and the library's shared SQL session helper.
    The bundled gateway loader stays with its existing host-owned installer.
    """
    db = globals().get("_SPUR_DUCKDB_CONNECTION")
    if db is None or db.sql("SELECT version()").fetchall()[0][0] not in ("v1.5.5", "v2.0.0"):
        return
    platform = db.sql("PRAGMA platform").fetchall()[0][0]
    current = globals().get("_SPUR_DUCKDB_RUNTIME")
    if current is not None and current.db is not db:
        current = None
    for name in ("json", "parquet", "httpfs"):
        if current is None or not current._entry(name)["loaded"]:
            _spur_duckdb_job(platform, name, "load")


class _SpurJevChannel:
    def __init__(self, descriptor):
        import socket
        self.socket = socket.socket(fileno=int(descriptor))
        self.socket.set_inheritable(False)
        self.lease = self._frame()

    def _read(self, count):
        chunks = bytearray()
        while len(chunks) < count:
            chunk = self.socket.recv(count - len(chunks))
            if not chunk:
                raise RuntimeError("Jev private credential channel closed")
            chunks.extend(chunk)
        return bytes(chunks)

    def _frame(self):
        return self._read(int.from_bytes(self._read(4), "big"))

    def _authorize(self, resolve):
        self.socket.sendall(len(self.lease).to_bytes(4, "big") + self.lease + bytes([resolve]))
        status = self._read(1)
        if status == b"\x02":
            return False
        if status != b"\x01":
            raise RuntimeError("denied")
        return True

    def available(self):
        try:
            return self._authorize(False)
        except Exception:
            self.socket.close()
            raise RuntimeError("Jev credential request denied or unavailable. Verify and enable the current managed local kernel in Settings.") from None

    def request(self):
        try:
            if not self._authorize(True):
                return None
            token = self._frame().decode("ascii")
            if not token or any(not 33 <= ord(c) <= 126 for c in token):
                raise ValueError("invalid credential")
            return token
        except Exception:
            self.socket.close()
            raise RuntimeError("Jev credential request denied or unavailable. Verify and enable the current managed local kernel in Settings.") from None


# Repeated generated SQL preambles preserve the captured channel. Failed
# attachment remains failed until kernel restart; never retry a legacy file.
if "_SPUR_JEV_BROKER" not in globals():
    _SPUR_JEV_BROKER = None
    _SPUR_JEV_BROKER_FAILED = False


def _spur_jev_channel():
    import os
    global _SPUR_JEV_BROKER, _SPUR_JEV_BROKER_FAILED
    if _SPUR_JEV_BROKER_FAILED:
        raise RuntimeError("Jev private credential channel is unavailable. Restart the managed local kernel.")
    descriptor = os.environ.pop("SPUR_JEV_BROKER_FD", None)
    if descriptor is not None:
        try:
            if _SPUR_JEV_BROKER is not None:
                raise RuntimeError("duplicate channel")
            _SPUR_JEV_BROKER = _SpurJevChannel(descriptor)
        except Exception:
            _SPUR_JEV_BROKER_FAILED = True
            raise RuntimeError("Jev private credential channel is unavailable. Restart the managed local kernel.") from None
    return _SPUR_JEV_BROKER


def _spur_close_jev_channel():
    """Disable/revoke/replacement is terminal for this inherited descriptor."""
    import os
    global _SPUR_JEV_BROKER_FAILED
    _SPUR_JEV_BROKER_FAILED = True
    if _SPUR_JEV_BROKER is not None:
        _SPUR_JEV_BROKER.socket.close()
    descriptor = os.environ.pop("SPUR_JEV_BROKER_FD", None)
    if descriptor is not None:
        try:
            os.close(int(descriptor))
        except (ValueError, OSError):
            pass


class _SpurJevSession:
    def __init__(self, db):
        self.db = db
        self.token = None
        self.previous = None

    def __enter__(self):
        import os
        _spur_duckdb_baseline()
        runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
        connection = _spur_managed_connection(self.db)
        if runtime is None or runtime.db is not connection:
            return self
        entry = runtime._entry("jev")
        if not (entry["desired_enabled"] and entry["loaded"] and entry["signature_verified"]
                and "jev" in runtime.loaded):
            return self
        try:
            channel = _spur_jev_channel()
            if channel is None or not channel.available():
                entry["restart_required"] = True
                entry["activation_transport_restart"] = True
                entry["reason"] = "restart_required"
                return self
        except Exception:
            # A dead/revoked channel cannot renew in-place. Do not block ordinary
            # SQL, read a legacy file or install/load anything as a fallback.
            entry["restart_required"] = True
            entry["activation_transport_restart"] = True
            entry["reason"] = "restart_required"
            return self
        endpoint = self.db.sql("SELECT current_setting('jev_api_url')").fetchall()[0][0]
        if endpoint != "https://api.typesafe.ai/v1/systemone":
            raise RuntimeError("Managed Jev credentials require the TypeSafe HTTPS endpoint. Reset jev_api_url before running this cell.")
        self.db.sql("RESET jev_api_key")
        # Resolve only after installation and exact endpoint validation. The
        # broker rechecks lease, enablement and generation on every request.
        try:
            token = channel.request()
        except Exception:
            entry["restart_required"] = True
            entry["reason"] = "restart_required"
            return self
        if token is None:  # disabled while setup was in flight
            return self
        self.previous = os.environ.get("TYPESAFE_API_KEY")
        self.token = token
        os.environ["TYPESAFE_API_KEY"] = token
        return self

    def __exit__(self, error_type, error, traceback):
        import os
        import traceback as traceback_module

        if self.token is None:
            return False
        token = self.token
        self.token = None
        if self.previous is None:
            os.environ.pop("TYPESAFE_API_KEY", None)
        else:
            os.environ["TYPESAFE_API_KEY"] = self.previous
        self.previous = None
        # A remote API can echo a header in an error. Suppress the original
        # traceback chain and remove the exact managed token from the diagnostic.
        if error is not None and token in "".join(traceback_module.format_exception(error_type, error, traceback)):
            message = str(error).replace(token, "[redacted]")
            token = None
            error = None
            traceback = None
            raise RuntimeError(message) from None
        return False

def _spur_restore_catalog_extensions(connection, requirements):
    """Restore optional membership without making unrelated SQL depend on its success."""
    import sys
    failures = globals().setdefault("_SPUR_CATALOG_EXTENSION_FAILURES", {})
    errors = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        key = (id(connection), name, source)
        if _spur_catalog_extensions_ready(connection, [item]):
            failures.pop(key, None)
            continue
        previous = failures.get(key)
        if previous and previous[0] is connection:
            errors[name] = previous[1]
            continue
        try:
            _spur_prepare_catalog_extensions(connection, [item])
        except Exception as error:
            errors[name] = str(error)
            failures[key] = (connection, str(error))
            print(f"SPUR: optional extension {name} could not be restored: {error}. Retry or remove it in Extensions.", file=sys.stderr)
    return errors


def _spur_catalog_extensions_ready(connection, requirements):
    """Read current connection identity and proofs without installing or loading code."""
    import re
    rows = connection.sql(
        "SELECT extension_name, installed, loaded, install_mode, installed_from, "
        "install_path, aliases FROM duckdb_extensions()"
    ).fetchall()
    inventory = {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}
    proofs = globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name) or source not in ("core", "community"):
            return False
        row = inventory.get(name)
        if not row or not row[2]:
            return False
        if row[3] == "STATICALLY_LINKED" and source == "core":
            continue
        proof = proofs.get((id(connection), row[0]))
        if not proof or proof[0] is not connection or proof[1] != source:
            return False
    return True


def _spur_prepare_connection_extensions(statements):
    """Use the same verifier for host-generated connection INSTALL/LOAD pairs."""
    import re
    requirements, loads = {}, set()
    for statement in statements:
        install = re.fullmatch(r"INSTALL ([a-z][a-z0-9_]*)(?: FROM (core|community))?", statement)
        load = re.fullmatch(r"LOAD ([a-z][a-z0-9_]*)", statement)
        if install:
            name, source = install.group(1), install.group(2) or "core"
            if name in requirements and requirements[name] != source:
                raise ValueError("Conflicting managed connection repositories")
            requirements[name] = source
        elif load:
            loads.add(load.group(1))
        else:
            raise ValueError("Invalid managed connection extension statement")
    if loads != set(requirements):
        raise ValueError("Managed connection extension installation and load must match")
    connection = globals().get("_SPUR_DUCKDB_CONNECTION")
    if connection is None:
        connection = duckdb.default_connection() if hasattr(duckdb, "default_connection") else duckdb
    _spur_prepare_catalog_extensions(connection, [
        {"packageName": name, "source": source} for name, source in requirements.items()
    ])


def _spur_prepare_catalog_extensions(connection, requirements):
    """Prepare signed packages on the notebook's shared DuckDB connection."""
    import duckdb as engine
    import hashlib
    from pathlib import Path
    import re
    import shutil
    import tempfile

    checked = {}
    for item in requirements:
        name, source = item.get("packageName", ""), item.get("source", "")
        if not re.fullmatch(r"[a-z][a-z0-9_]*", name):
            raise ValueError("Invalid managed DuckDB extension identity")
        if source not in ("core", "community"):
            raise ValueError(
                "Managed catalog setup supports only Core and Community repositories; "
                "other repositories require explicit setup in notebook SQL"
            )
        if name in checked and checked[name] != source:
            raise ValueError("One package cannot use two repositories in the same runtime")
        checked[name] = source
        # Explicit preparation is an intentional retry; ordinary SQL uses restore's memo.
        globals().get("_SPUR_CATALOG_EXTENSION_FAILURES", {}).pop((id(connection), name, source), None)

    def inventory(db):
        rows = db.sql(
            "SELECT extension_name, installed, loaded, install_mode, installed_from, "
            "install_path, aliases FROM duckdb_extensions()"
        ).fetchall()
        return {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}

    def matches(row, source):
        origin = row[4] or ""
        repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
        return origin == source or origin.rstrip("/") in (
            "http://" + repository, "https://" + repository)

    # Proofs belong to this exact connection, surviving redefinition on SQL replay.
    # Retain private immutable copies for the lifetime of the kernel.
    proofs = globals().setdefault("_SPUR_CATALOG_EXTENSION_PROOFS", {})
    for name, source in checked.items():
        row = inventory(connection).get(name)
        if row and row[3] == "STATICALLY_LINKED" and source == "core":
            if not row[2]:
                connection.load_extension(name)
            continue
        proof = proofs.get((id(connection), row[0] if row else name))
        if row and row[2]:
            # The required HTTPFS baseline has an existing signed verifier and
            # frozen copy. Share that exact proof, never a bare loaded flag or
            # repository label. This also supports already-running baseline code.
            if proof is None and row[0] == "httpfs" and source == "core":
                runtime = globals().get("_SPUR_DUCKDB_RUNTIME")
                if runtime is not None and runtime.db is connection and "httpfs" in runtime.loaded:
                    baseline = runtime._entry("httpfs")
                    private = runtime.verified.get("httpfs")
                    if baseline.get("signature_verified") is True and baseline.get("loaded") is True and private is not None:
                        try:
                            digest = hashlib.sha256((Path(private.name) / "httpfs.duckdb_extension").read_bytes()).digest()
                            if digest.hex() == baseline.get("artifact_digest"):
                                proof = (connection, source, private, digest)
                                proofs[(id(connection), row[0])] = proof
                        except OSError:
                            pass
            if proof and proof[0] is connection and proof[1] == source:
                continue
            raise RuntimeError(f"{name} is already loaded without this setup's verified artifact; restart the kernel and retry Install in notebook")

        # The shared connection permits the app's bundled unsigned gateway.
        # Always verify third-party bytes in an isolated engine with signing ON.
        # Same-user/kernel arbitrary code remains trusted; this is not a sandbox.
        config = {
            "allow_unsigned_extensions": "false",
            "allow_extensions_metadata_mismatch": "false",
            "allow_community_extensions": "true" if source == "community" else "false",
            "autoload_known_extensions": "false",
            "autoinstall_known_extensions": "false",
        }
        version = connection.sql("SELECT version()").fetchall()[0][0]
        repository_aware = version == "v2.0.0" and source == "community"
        selected_requires_named_load = repository_aware and connection.sql(
            "SELECT current_setting('allow_unsigned_extensions')"
        ).fetchall()[0][0] is False
        if version == "v2.0.0":
            directories = connection.sql("SELECT current_setting('extension_directories')").fetchall()[0][0]
            if directories:
                config["extension_directories"] = directories
            repository_dir = connection.sql("SELECT current_setting('extension_repository_directory')").fetchall()[0][0]
            if repository_dir:
                config["extension_repository_directory"] = repository_dir
        else:
            config["home_directory"] = connection.sql("SELECT current_setting('home_directory')").fetchall()[0][0]
            directory = connection.sql("SELECT current_setting('extension_directory')").fetchall()[0][0]
            if directory:
                config["extension_directory"] = directory
        private = None
        try:
            with engine.connect(config=config) as verifier:
                for query in ("SELECT version()", "PRAGMA platform"):
                    if verifier.sql(query).fetchall() != connection.sql(query).fetchall():
                        raise RuntimeError("DuckDB runtime changed during extension verification")
                artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    if repository_aware:
                        verifier.execute(f"INSTALL {name} FROM {source}").fetchall()
                    else:
                        repository = "extensions.duckdb.org" if source == "core" else "community-extensions.duckdb.org"
                        verifier.install_extension(name, repository_url="http://" + repository)
                    artifact = inventory(verifier).get(name)
                if not artifact or not artifact[1]:
                    raise RuntimeError(f"{name} installation is unavailable for this DuckDB runtime")
                if artifact[3] != "REPOSITORY" or not matches(artifact, source):
                    raise RuntimeError(f"{name} is cached from another repository; resolve its source before retrying")
                private = tempfile.TemporaryDirectory(prefix="spur-duckdb-catalog-")
                directory = Path(private.name)
                if repository_aware:
                    platform = verifier.sql("PRAGMA platform").fetchall()[0][0]
                    directory = directory / version / platform
                    directory.mkdir(parents=True)
                path = directory / (artifact[0] + ".duckdb_extension")
                shutil.copyfile(artifact[5], path)
                path.chmod(0o400)
                digest = hashlib.sha256(path.read_bytes()).digest()
                if repository_aware:
                    info = Path(artifact[5] + ".info")
                    if info.is_file():
                        shutil.copyfile(info, Path(str(path) + ".info"))
                    # Only this isolated database changes its lookup directory.
                    # Named LOAD retains DuckDB 2.0's repository signing keys.
                    verifier.execute("SET extension_directories = ?", [[private.name]]).fetchall()
                    verifier.execute(f"LOAD {name} FROM {source}").fetchall()
                else:
                    verifier.load_extension(str(path))  # signature and ABI checked by DuckDB
                if hashlib.sha256(path.read_bytes()).digest() != digest:
                    raise RuntimeError("Extension artifact changed during verification")
            snapshot = globals().get("_spur_snapshot_function_registration")
            before = snapshot(connection) if callable(snapshot) else None
            if hashlib.sha256(path.read_bytes()).digest() != digest:
                raise RuntimeError("Extension artifact changed during verification")
            if selected_requires_named_load:
                # A strict DuckDB 2.0 host also needs the repository identity.
                # It independently checks signing on the selected connection.
                if hashlib.sha256(Path(artifact[5]).read_bytes()).digest() != digest:
                    raise RuntimeError("Extension artifact changed during verification")
                connection.execute(f"LOAD {name} FROM {source}").fetchall()
                if hashlib.sha256(Path(artifact[5]).read_bytes()).digest() != digest:
                    raise RuntimeError("Extension artifact changed during verification")
            else:
                connection.load_extension(str(path))  # load exactly the verified bytes
            observed = inventory(connection).get(name)
            if not observed or not observed[2]:
                raise RuntimeError(f"{name} did not load into the notebook connection")
            proofs[(id(connection), observed[0])] = (connection, source, private, digest)
            record = globals().get("_spur_record_function_ownership")
            if callable(record):
                record(connection, observed[0], before)
            private = None
        finally:
            if private is not None:
                private.cleanup()


_spur_duckdb_baseline()
# SPUR DuckDB readiness comm v1
def _spur_register_readiness_comm(verifier_source):
    """Answer host observations without starting another notebook cell."""
    def on_open(comm, message):
        try:
            requirements = message.get("content", {}).get("data", {}).get("requirements", [])
            connection = globals().get("_SPUR_DUCKDB_CONNECTION")
            scope = {"_SPUR_CATALOG_EXTENSION_PROOFS": globals().get("_SPUR_CATALOG_EXTENSION_PROOFS", {})}
            exec(verifier_source, scope)
            ready = bool(connection is not None and scope["_spur_catalog_extensions_ready"](connection, requirements))
            data = {"protocol": "spur.duckdb.readiness.v1", "ready": ready, "connection_id": str(id(connection)) if connection is not None else None}
        except Exception:
            # The channel exists, but the current connection cannot be checked.
            data = {"protocol": "spur.duckdb.readiness.v1", "ready": False, "connection_id": None}
        comm.close(data=data)

    try:
        get_ipython().kernel.comm_manager.register_target("spur.duckdb.readiness.v1", on_open)
    except Exception:
        pass

_spur_register_readiness_comm("def _spur_restore_catalog_extensions(connection, requirements):\n    \"\"\"Restore optional membership without making unrelated SQL depend on its success.\"\"\"\n    import sys\n    failures = globals().setdefault(\"_SPUR_CATALOG_EXTENSION_FAILURES\", {})\n    errors = {}\n    for item in requirements:\n        name, source = item.get(\"packageName\", \"\"), item.get(\"source\", \"\")\n        key = (id(connection), name, source)\n        if _spur_catalog_extensions_ready(connection, [item]):\n            failures.pop(key, None)\n            continue\n        previous = failures.get(key)\n        if previous and previous[0] is connection:\n            errors[name] = previous[1]\n            continue\n        try:\n            _spur_prepare_catalog_extensions(connection, [item])\n        except Exception as error:\n            errors[name] = str(error)\n            failures[key] = (connection, str(error))\n            print(f\"SPUR: optional extension {name} could not be restored: {error}. Retry or remove it in Extensions.\", file=sys.stderr)\n    return errors\n\n\ndef _spur_catalog_extensions_ready(connection, requirements):\n    \"\"\"Read current connection identity and proofs without installing or loading code.\"\"\"\n    import re\n    rows = connection.sql(\n        \"SELECT extension_name, installed, loaded, install_mode, installed_from, \"\n        \"install_path, aliases FROM duckdb_extensions()\"\n    ).fetchall()\n    inventory = {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}\n    proofs = globals().get(\"_SPUR_CATALOG_EXTENSION_PROOFS\", {})\n    for item in requirements:\n        name, source = item.get(\"packageName\", \"\"), item.get(\"source\", \"\")\n        if not re.fullmatch(r\"[a-z][a-z0-9_]*\", name) or source not in (\"core\", \"community\"):\n            return False\n        row = inventory.get(name)\n        if not row or not row[2]:\n            return False\n        if row[3] == \"STATICALLY_LINKED\" and source == \"core\":\n            continue\n        proof = proofs.get((id(connection), row[0]))\n        if not proof or proof[0] is not connection or proof[1] != source:\n            return False\n    return True\n\n\ndef _spur_prepare_connection_extensions(statements):\n    \"\"\"Use the same verifier for host-generated connection INSTALL/LOAD pairs.\"\"\"\n    import re\n    requirements, loads = {}, set()\n    for statement in statements:\n        install = re.fullmatch(r\"INSTALL ([a-z][a-z0-9_]*)(?: FROM (core|community))?\", statement)\n        load = re.fullmatch(r\"LOAD ([a-z][a-z0-9_]*)\", statement)\n        if install:\n            name, source = install.group(1), install.group(2) or \"core\"\n            if name in requirements and requirements[name] != source:\n                raise ValueError(\"Conflicting managed connection repositories\")\n            requirements[name] = source\n        elif load:\n            loads.add(load.group(1))\n        else:\n            raise ValueError(\"Invalid managed connection extension statement\")\n    if loads != set(requirements):\n        raise ValueError(\"Managed connection extension installation and load must match\")\n    connection = globals().get(\"_SPUR_DUCKDB_CONNECTION\")\n    if connection is None:\n        connection = duckdb.default_connection() if hasattr(duckdb, \"default_connection\") else duckdb\n    _spur_prepare_catalog_extensions(connection, [\n        {\"packageName\": name, \"source\": source} for name, source in requirements.items()\n    ])\n\n\ndef _spur_prepare_catalog_extensions(connection, requirements):\n    \"\"\"Prepare signed packages on the notebook's shared DuckDB connection.\"\"\"\n    import duckdb as engine\n    import hashlib\n    from pathlib import Path\n    import re\n    import shutil\n    import tempfile\n\n    checked = {}\n    for item in requirements:\n        name, source = item.get(\"packageName\", \"\"), item.get(\"source\", \"\")\n        if not re.fullmatch(r\"[a-z][a-z0-9_]*\", name):\n            raise ValueError(\"Invalid managed DuckDB extension identity\")\n        if source not in (\"core\", \"community\"):\n            raise ValueError(\n                \"Managed catalog setup supports only Core and Community repositories; \"\n                \"other repositories require explicit setup in notebook SQL\"\n            )\n        if name in checked and checked[name] != source:\n            raise ValueError(\"One package cannot use two repositories in the same runtime\")\n        checked[name] = source\n        # Explicit preparation is an intentional retry; ordinary SQL uses restore's memo.\n        globals().get(\"_SPUR_CATALOG_EXTENSION_FAILURES\", {}).pop((id(connection), name, source), None)\n\n    def inventory(db):\n        rows = db.sql(\n            \"SELECT extension_name, installed, loaded, install_mode, installed_from, \"\n            \"install_path, aliases FROM duckdb_extensions()\"\n        ).fetchall()\n        return {alias: row for row in rows for alias in [row[0], *(row[6] or [])]}\n\n    def matches(row, source):\n        origin = row[4] or \"\"\n        repository = \"extensions.duckdb.org\" if source == \"core\" else \"community-extensions.duckdb.org\"\n        return origin == source or origin.rstrip(\"/\") in (\n            \"http://\" + repository, \"https://\" + repository)\n\n    # Proofs belong to this exact connection, surviving redefinition on SQL replay.\n    # Retain private immutable copies for the lifetime of the kernel.\n    proofs = globals().setdefault(\"_SPUR_CATALOG_EXTENSION_PROOFS\", {})\n    for name, source in checked.items():\n        row = inventory(connection).get(name)\n        if row and row[3] == \"STATICALLY_LINKED\" and source == \"core\":\n            if not row[2]:\n                connection.load_extension(name)\n            continue\n        proof = proofs.get((id(connection), row[0] if row else name))\n        if row and row[2]:\n            # The required HTTPFS baseline has an existing signed verifier and\n            # frozen copy. Share that exact proof, never a bare loaded flag or\n            # repository label. This also supports already-running baseline code.\n            if proof is None and row[0] == \"httpfs\" and source == \"core\":\n                runtime = globals().get(\"_SPUR_DUCKDB_RUNTIME\")\n                if runtime is not None and runtime.db is connection and \"httpfs\" in runtime.loaded:\n                    baseline = runtime._entry(\"httpfs\")\n                    private = runtime.verified.get(\"httpfs\")\n                    if baseline.get(\"signature_verified\") is True and baseline.get(\"loaded\") is True and private is not None:\n                        try:\n                            digest = hashlib.sha256((Path(private.name) / \"httpfs.duckdb_extension\").read_bytes()).digest()\n                            if digest.hex() == baseline.get(\"artifact_digest\"):\n                                proof = (connection, source, private, digest)\n                                proofs[(id(connection), row[0])] = proof\n                        except OSError:\n                            pass\n            if proof and proof[0] is connection and proof[1] == source:\n                continue\n            raise RuntimeError(f\"{name} is already loaded without this setup's verified artifact; restart the kernel and retry Install in notebook\")\n\n        # The shared connection permits the app's bundled unsigned gateway.\n        # Always verify third-party bytes in an isolated engine with signing ON.\n        # Same-user/kernel arbitrary code remains trusted; this is not a sandbox.\n        config = {\n            \"allow_unsigned_extensions\": \"false\",\n            \"allow_extensions_metadata_mismatch\": \"false\",\n            \"allow_community_extensions\": \"true\" if source == \"community\" else \"false\",\n            \"autoload_known_extensions\": \"false\",\n            \"autoinstall_known_extensions\": \"false\",\n        }\n        version = connection.sql(\"SELECT version()\").fetchall()[0][0]\n        repository_aware = version == \"v2.0.0\" and source == \"community\"\n        selected_requires_named_load = repository_aware and connection.sql(\n            \"SELECT current_setting('allow_unsigned_extensions')\"\n        ).fetchall()[0][0] is False\n        if version == \"v2.0.0\":\n            directories = connection.sql(\"SELECT current_setting('extension_directories')\").fetchall()[0][0]\n            if directories:\n                config[\"extension_directories\"] = directories\n            repository_dir = connection.sql(\"SELECT current_setting('extension_repository_directory')\").fetchall()[0][0]\n            if repository_dir:\n                config[\"extension_repository_directory\"] = repository_dir\n        else:\n            config[\"home_directory\"] = connection.sql(\"SELECT current_setting('home_directory')\").fetchall()[0][0]\n            directory = connection.sql(\"SELECT current_setting('extension_directory')\").fetchall()[0][0]\n            if directory:\n                config[\"extension_directory\"] = directory\n        private = None\n        try:\n            with engine.connect(config=config) as verifier:\n                for query in (\"SELECT version()\", \"PRAGMA platform\"):\n                    if verifier.sql(query).fetchall() != connection.sql(query).fetchall():\n                        raise RuntimeError(\"DuckDB runtime changed during extension verification\")\n                artifact = inventory(verifier).get(name)\n                if not artifact or not artifact[1]:\n                    if repository_aware:\n                        verifier.execute(f\"INSTALL {name} FROM {source}\").fetchall()\n                    else:\n                        repository = \"extensions.duckdb.org\" if source == \"core\" else \"community-extensions.duckdb.org\"\n                        verifier.install_extension(name, repository_url=\"http://\" + repository)\n                    artifact = inventory(verifier).get(name)\n                if not artifact or not artifact[1]:\n                    raise RuntimeError(f\"{name} installation is unavailable for this DuckDB runtime\")\n                if artifact[3] != \"REPOSITORY\" or not matches(artifact, source):\n                    raise RuntimeError(f\"{name} is cached from another repository; resolve its source before retrying\")\n                private = tempfile.TemporaryDirectory(prefix=\"spur-duckdb-catalog-\")\n                directory = Path(private.name)\n                if repository_aware:\n                    platform = verifier.sql(\"PRAGMA platform\").fetchall()[0][0]\n                    directory = directory / version / platform\n                    directory.mkdir(parents=True)\n                path = directory / (artifact[0] + \".duckdb_extension\")\n                shutil.copyfile(artifact[5], path)\n                path.chmod(0o400)\n                digest = hashlib.sha256(path.read_bytes()).digest()\n                if repository_aware:\n                    info = Path(artifact[5] + \".info\")\n                    if info.is_file():\n                        shutil.copyfile(info, Path(str(path) + \".info\"))\n                    # Only this isolated database changes its lookup directory.\n                    # Named LOAD retains DuckDB 2.0's repository signing keys.\n                    verifier.execute(\"SET extension_directories = ?\", [[private.name]]).fetchall()\n                    verifier.execute(f\"LOAD {name} FROM {source}\").fetchall()\n                else:\n                    verifier.load_extension(str(path))  # signature and ABI checked by DuckDB\n                if hashlib.sha256(path.read_bytes()).digest() != digest:\n                    raise RuntimeError(\"Extension artifact changed during verification\")\n            snapshot = globals().get(\"_spur_snapshot_function_registration\")\n            before = snapshot(connection) if callable(snapshot) else None\n            if hashlib.sha256(path.read_bytes()).digest() != digest:\n                raise RuntimeError(\"Extension artifact changed during verification\")\n            if selected_requires_named_load:\n                # A strict DuckDB 2.0 host also needs the repository identity.\n                # It independently checks signing on the selected connection.\n                if hashlib.sha256(Path(artifact[5]).read_bytes()).digest() != digest:\n                    raise RuntimeError(\"Extension artifact changed during verification\")\n                connection.execute(f\"LOAD {name} FROM {source}\").fetchall()\n                if hashlib.sha256(Path(artifact[5]).read_bytes()).digest() != digest:\n                    raise RuntimeError(\"Extension artifact changed during verification\")\n            else:\n                connection.load_extension(str(path))  # load exactly the verified bytes\n            observed = inventory(connection).get(name)\n            if not observed or not observed[2]:\n                raise RuntimeError(f\"{name} did not load into the notebook connection\")\n            proofs[(id(connection), observed[0])] = (connection, source, private, digest)\n            record = globals().get(\"_spur_record_function_ownership\")\n            if callable(record):\n                record(connection, observed[0], before)\n            private = None\n        finally:\n            if private is not None:\n                private.cleanup()\n")
def _spur_reconcile_managed_datasources(desired_views, removed_views, desired_aliases, removed_aliases):
    # Only objects emitted by managed setup belong to this registry. Never drop
    # a whole schema: users may have created their own objects alongside ours.
    def key(value):
        return value.translate(str.maketrans('ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'))

    def ident(value):
        return '"' + value.replace('"', '""') + '"'

    def views(values):
        return {(key(schema), key(name)): (schema, name) for schema, name in values}

    wanted_views = views(desired_views)
    previous_views = views(globals().get('_SPUR_MANAGED_VIEWS', []))
    previous_views.update(views(removed_views))
    for identity, (schema, name) in previous_views.items():
        if identity not in wanted_views:
            duckdb.sql('DROP VIEW IF EXISTS "memory".' + ident(schema) + '.' + ident(name))

    wanted_aliases = {key(name) for name in desired_aliases}
    previous_aliases = set(globals().get('_SPUR_MANAGED_ALIASES', [])) | set(removed_aliases)
    attached = {key(row[0]): row[0] for row in duckdb.sql('SELECT database_name FROM duckdb_databases()').fetchall()}
    for name in previous_aliases:
        identity = key(name)
        if identity in wanted_aliases or identity in {'memory', 'temp', 'system'}:
            continue
        if identity in attached:
            duckdb.sql('DETACH ' + ident(attached[identity]))
        for registry_name in ('_SPUR_REMOTE_CONNECTIONS', '_SPUR_LAKEHOUSE_CONNECTIONS'):
            registry = globals().get(registry_name, {})
            for cached in list(registry):
                if key(cached) == identity:
                    del registry[cached]
        for cached in list(globals().get('_SPUR_REMOTE_ATTACHED', set())):
            if key(cached) == identity:
                _SPUR_REMOTE_ATTACHED.discard(cached)

    globals()['_SPUR_MANAGED_VIEWS'] = list(wanted_views.values())
    globals()['_SPUR_MANAGED_ALIASES'] = list(desired_aliases)

_spur_reconcile_managed_datasources([["github-pat","security_advisories"],["github-pat","authenticated_repos"],["github-pat","user_issues"],["github-pat","user_orgs"],["github-pat","user_gists"],["github-pat","starred"],["github-pat","notifications"],["github-pat","user_followers"],["github-pat","user_emails"],["github-pat","authenticated_user"],["github-pat","user_following"]], [], [], [])
try:
    duckdb.sql("SET search_path = 'memory'")
except Exception:
    pass
if not globals().get("_SPUR_REST_EXTENSION_LOADED", False):
    if _SPUR_DUCKDB_GATEWAY_COMPATIBLE:
        try:
            duckdb.sql(f"LOAD '{_SPUR_DUCKDB_EXTENSION_SQL}'")
            _SPUR_REST_EXTENSION_LOADED = True
        except Exception as _spur_rest_load_err:
            _SPUR_REST_EXTENSION_LOADED = False
            print(
                f"SPUR: failed to LOAD spur_rest extension ({_SPUR_DUCKDB_EXTENSION_PATH}): {_spur_rest_load_err}",
                file=sys.stderr,
            )
_spur_live_api_tables = duckdb.sql("SELECT count(*) FROM duckdb_functions() WHERE function_name = 'spur_rest_table'").fetchall()[0][0] > 0
_spur_native_sources = set()
if duckdb.sql("SELECT count(*) FROM duckdb_functions() WHERE function_name = 'spur_native_refresh'").fetchall()[0][0]:
    for _spur_source in ["github","github-pat"]:
        for _spur_owner, _spur_sql, _spur_revision, _spur_error in duckdb.execute("SELECT * FROM spur_native_refresh(?)", [_spur_source]).fetchall():
            _spur_native_sources.update((_spur_source, _spur_owner))
            if _spur_error:
                print(f"SPUR: native source {_spur_owner} refresh failed: {_spur_error}", file=sys.stderr)
                continue
            try:
                duckdb.execute(_spur_sql)
            except Exception as _spur_refresh_error:
                duckdb.execute("ROLLBACK")
                print(f"SPUR: native source {_spur_owner} publication failed: {_spur_refresh_error}", file=sys.stderr)
try:
    _SPUR_REST_TF_NAMES = {
        r[0]
        for r in duckdb.sql(
            "SELECT DISTINCT function_name FROM duckdb_functions() WHERE function_name LIKE '%\\_%' ESCAPE '\\'"
        ).fetchall()
    }
except Exception as _spur_tf_snap_err:
    _SPUR_REST_TF_NAMES = set()
    print(
        f"SPUR: could not snapshot table functions: {_spur_tf_snap_err}",
        file=sys.stderr,
    )
duckdb.sql("CREATE SCHEMA IF NOT EXISTS \"github-pat\"") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"security_advisories\" AS SELECT * FROM spur_rest_table('github', 'security_advisories')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"security_advisories\" AS SELECT * FROM \"github_security_advisories\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"authenticated_repos\" AS SELECT * FROM spur_rest_table('github', 'authenticated_repos')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"authenticated_repos\" AS SELECT * FROM \"github_authenticated_repos\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"user_issues\" AS SELECT * FROM spur_rest_table('github', 'user_issues')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"user_issues\" AS SELECT * FROM \"github_user_issues\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"user_orgs\" AS SELECT * FROM spur_rest_table('github', 'user_orgs')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"user_orgs\" AS SELECT * FROM \"github_user_orgs\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"user_gists\" AS SELECT * FROM spur_rest_table('github', 'user_gists')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"user_gists\" AS SELECT * FROM \"github_user_gists\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"starred\" AS SELECT * FROM spur_rest_table('github', 'starred')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"starred\" AS SELECT * FROM \"github_starred\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"notifications\" AS SELECT * FROM spur_rest_table('github', 'notifications')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"notifications\" AS SELECT * FROM \"github_notifications\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"user_followers\" AS SELECT * FROM spur_rest_table('github', 'user_followers')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"user_followers\" AS SELECT * FROM \"github_user_followers\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"user_emails\" AS SELECT * FROM spur_rest_table('github', 'user_emails')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"user_emails\" AS SELECT * FROM \"github_user_emails\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"authenticated_user\" AS SELECT * FROM spur_rest_table('github', 'authenticated_user')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"authenticated_user\" AS SELECT * FROM \"github_authenticated_user\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None
duckdb.sql("CREATE OR REPLACE VIEW \"github-pat\".\"user_following\" AS SELECT * FROM spur_rest_table('github', 'user_following')" if _spur_live_api_tables else "CREATE OR REPLACE VIEW \"github-pat\".\"user_following\" AS SELECT * FROM \"github_user_following\"()") if "github-pat" not in _spur_native_sources and "github" not in _spur_native_sources else None


CountBIGINT


In [5]:
-- Read-only, tightly scoped inbox query through the saved @github-pat connection.
SELECT number,title,state,user_login,html_url,updated_at
FROM github_search_issues(
  "q" = 'repo:duckdb/duckdb is:pr 28731',
  "sort" = 'updated',
  "order" = 'desc')
ORDER BY updated_at DESC LIMIT 5;

number,title,state,user_login,html_url,updated_at
28731,Preserve aggregate domains for proven singleton groups,open,kevintruong,https://github.com/duckdb/duckdb/pull/28731,2026-11-17T15:22:12Z


In [6]:
-- A narrow real inbox slice. Search limits are not a claim of a complete repository inbox.
CREATE OR REPLACE TABLE github_poc_pr_inbox AS
SELECT number,title,state,user_login,html_url,updated_at
FROM github_search_issues("q" = 'repo:duckdb/duckdb is:pr 28731',"sort"='updated',"order"='desc');
CREATE OR REPLACE TABLE github_poc_issue_inbox AS
SELECT number,title,state,user_login,html_url,updated_at
FROM github_search_issues("q" = 'repo:duckdb/duckdb is:issue 28615',"sort"='updated',"order"='desc');
SELECT 'pull_request' AS item_type,* FROM github_poc_pr_inbox
UNION ALL
SELECT 'issue' AS item_type,* FROM github_poc_issue_inbox
ORDER BY item_type,number;

item_type,number,title,state,user_login,html_url,updated_at
issue,28615,Performance regression after #28456: DISTINCT ON loses join-key statistics and changes join order,open,ted-wq-x,https://github.com/duckdb/duckdb/issues/28615,2026-11-11T17:09:37Z
pull_request,28731,Preserve aggregate domains for proven singleton groups,open,kevintruong,https://github.com/duckdb/duckdb/pull/28731,2026-11-17T15:22:12Z


## GitHub inbox result and collection boundary

The SQL cells above query the attached **@github-pat** connection. They retrieve real PR and issue records without copying credentials into a cell.

The saved search table exposes titles, state, author and timestamps. The saved list-pulls schema exposes branch names, but not immutable commit SHAs or detailed review/file/check resources. This is a real collector coverage gap for the future app.

For the PR comparison below, a separate read-only GitHub connector query supplies current base/head metadata. That captured response is stored with its provenance. GitHub inbox data and local Git evidence are joined by repository/PR and an explicit pinned comparison, rather than assuming branch names identify source bytes.


In [7]:
# Inspect integration state without making another GitHub request.
print('Inbox tables:',rows("SELECT table_name FROM information_schema.tables WHERE table_name LIKE 'github_poc_%' ORDER BY table_name"))
print('GitHub native function registration:',rows("SELECT function_name FROM duckdb_functions() WHERE function_name='github_search_issues'"))
print('Active graph grammar:',rows("SELECT current_setting('active_grammar_extensions') AS grammar"))


Inbox tables: [{'table_name': 'github_poc_issue_inbox'}, {'table_name': 'github_poc_pr_inbox'}]
GitHub native function registration: [{'function_name': 'github_search_issues'}]
Active graph grammar: [{'grammar': ['duckpgq']}]


In [8]:
# Isolate whether the failed two-resource SQL batch is a gateway or DuckDB limitation.
print('Registered signature:',rows("""
 SELECT function_name,function_type,parameters,parameter_types
 FROM duckdb_functions() WHERE function_name='github_search_issues'
"""))
gateway_probes = []
try:
    materialized = rows("""
      CREATE OR REPLACE TABLE github_poc_pr_inbox AS
      SELECT number,title,state,user_login,html_url,updated_at
      FROM github_search_issues("q"='repo:duckdb/duckdb is:pr 28731',"sort"='updated',"order"='desc')
    """)
    gateway_probes.append({'probe':'single CTAS through Python connection','status':'ok','result':materialized})
except Exception as error:
    gateway_probes.append({'probe':'single CTAS through Python connection','status':'failed',
                           'error_type':type(error).__name__,'message':str(error)[:700]})
save('github-materialization-probes',gateway_probes)
print(json.dumps(gateway_probes,indent=2))


Registered signature: [{'function_name': 'github_search_issues', 'function_type': 'table', 'parameters': ['order', 'q', 'sort'], 'parameter_types': ['VARCHAR', 'VARCHAR', 'VARCHAR']}]


[
  {
    "probe": "single CTAS through Python connection",
    "status": "ok",
    "result": [
      {
        "Count": 1
      }
    ]
  }
]


In [9]:
# Freeze source identity from the captured current PR metadata, independent of checkout.
PR = json.loads((ARTIFACT/'pr-context.json').read_text())
BASE, HEAD = PR['base_sha'], PR['head_sha']
REPO = '/Users/kevintruong/project/duckdb'
def git(*args):
    return subprocess.check_output(['git','-C',REPO,*args],timeout=30)
MERGE_BASE = git('merge-base',BASE,HEAD).decode().strip()
RUN_ID = hashlib.sha256(('duckdb/duckdb#28731:'+MERGE_BASE+':'+HEAD).encode()).hexdigest()[:20]
save('comparison',{'repo':'duckdb/duckdb','pr':28731,'target_base':BASE,'merge_base':MERGE_BASE,
                   'head':HEAD,'run_id':RUN_ID,'checkout_head':git('rev-parse','HEAD').decode().strip()})
pr_files = rows('SELECT * FROM git_diff_tree(?,?,?) ORDER BY file_path',[REPO,MERGE_BASE,HEAD])
save('changed-files',pr_files)
FILES = [r['file_path'] for r in pr_files if r['file_path'].endswith('.cpp')]
native_inputs,native_symbols,native_facts = [],[],[]
from collections import defaultdict
semantic_changes,raw_candidates = [],[]
for path in FILES:
    sides = {}
    for side,revision in [('before',MERGE_BASE),('after',HEAD)]:
        source = git('show',revision+':'+path)
        sha = hashlib.sha256(source).hexdigest()
        result = duckdb.execute('SELECT spur_graph_file_facts_result(?,?)',[path,source.decode()]).fetchall()[0][0]
        symbols = duckdb.execute('SELECT spur_graph_symbols(?,?)',[path,source.decode()]).fetchall()[0][0]
        assert result['status']=='ok',result['status']
        native_inputs.append({'file_path':path,'side':side,'revision':revision,'source_sha256':sha,
            'git_blob':git('rev-parse',revision+':'+path).decode().strip(),
            'status':result['status'],'symbols':len(symbols),
            'nodes':len(result['facts']['nodes']),'edges':len(result['facts']['edges'])})
        native_facts.append({'file_path':path,'side':side,'source_sha256':sha,'revision':revision,'facts':result['facts']})
        sides[side] = defaultdict(list)
        for s in symbols:
            item=dict(s,file_path=path,side=side,source_sha256=sha)
            item['body_sha256']=hashlib.sha256(source[s['byte_start']:s['byte_end']]).hexdigest()
            native_symbols.append(item)
            sides[side][(s['entity_name'],s['symbol_kind'],s['enclosing_scope'])].append(item)
    for key in set(sides['before'])|set(sides['after']):
        old,new=sides['before'].get(key,[]),sides['after'].get(key,[])
        if len(old)>1 or len(new)>1:
            continue
        if old and new and old[0]['body_sha256']==new[0]['body_sha256']:
            continue
        chosen=(new or old)[0]
        if chosen['symbol_kind'] in ('function','method'):
            raw_candidates.append(dict(chosen,change='added' if not old else 'deleted' if not new else 'modified'))
for c in raw_candidates:
    if not any(d['file_path']==c['file_path'] and d['entity_name']==c['entity_name']
               and d['byte_start']<=c['byte_start'] and d['byte_end']>=c['byte_end']
               and d['byte_end']-d['byte_start']>c['byte_end']-c['byte_start'] for d in raw_candidates):
        semantic_changes.append(c)
for name,data in [('native-inputs',native_inputs),('native-facts',native_facts),
                  ('native-symbols',native_symbols),('raw-change-candidates',raw_candidates),
                  ('semantic-changes',semantic_changes)]:
    save(name,data)
print('Comparison:',RUN_ID,MERGE_BASE[:12],HEAD[:12])
print('File diff:',[(r['file_path'],r['status']) for r in pr_files])
print('Native extraction:',native_inputs)
print('Semantic change seeds:',[c['entity_name'] for c in semantic_changes])
after_facts=[f for f in native_facts if f['side']=='after']
print('Node samples:',after_facts[0]['facts']['nodes'][:3])
print('Edge samples:',after_facts[1]['facts']['edges'][:3])


Comparison: cc714dc606372787dc8b 629e395a854d d29b26bb92ac
File diff: [('src/function/aggregate/distributive/first_last_any.cpp', 'modified'), ('src/optimizer/relation_statistics/relation_statistics_helper.cpp', 'modified'), ('test/configs/stats_dependent_tests.json', 'modified'), ('test/configs/verify_aggregate_state_export.json', 'modified'), ('test/sql/optimizer/statistics/distinct_on_stats_propagation.test', 'added')]
Native extraction: [{'file_path': 'src/function/aggregate/distributive/first_last_any.cpp', 'side': 'before', 'revision': '629e395a854d72dcb1f414e1a0451c68ac0e9f1f', 'source_sha256': '9845a113802478f2a80865a66186cda57e65cfa455314da1ff14c6e34ceb965a', 'git_blob': 'f36a4e8470537807470994713e41e2c4314116bb', 'status': 'ok', 'symbols': 82, 'nodes': 83, 'edges': 302}, {'file_path': 'src/function/aggregate/distributive/first_last_any.cpp', 'side': 'after', 'revision': 'd29b26bb92ac940a0807be0f43cc385b31623839', 'source_sha256': 'b8132604aa125a294008e7443936f6b034e71c536e886

In [10]:
# Convert actual native fact rows to globally scoped DuckPGQ vertices and resolved call edges.
# Native numeric IDs restart within each file-facts result, so never join them across files directly.
from collections import Counter
graph_nodes,call_edges,unresolved_calls = [],[],[]
profiles=[]
for file_result in after_facts:
    path,facts=file_result['file_path'],file_result['facts']
    spans={s['span_id']:s for s in facts['spans']}
    local={n['node_id']:n for n in facts['nodes']}
    canonical={}
    for node in facts['nodes']:
        span=spans.get(node['source_span_id'])
        containers=[]
        if span and node['kind'] in ('function','method'):
            for other in facts['nodes']:
                other_span=spans.get(other['source_span_id'])
                if other_span and other['label']==node['label'] and other['kind']==node['kind']:
                    if (other_span['start_byte']<=span['start_byte'] and other_span['end_byte']>=span['end_byte']
                        and other_span['end_byte']-other_span['start_byte']>span['end_byte']-span['start_byte']):
                        containers.append(other)
        canonical[node['node_id']]=min(containers,key=lambda n:n['node_id'])['node_id'] if containers else node['node_id']
    def global_id(local_id):
        return RUN_ID+':'+path+':'+local[canonical[local_id]]['stable_key']
    for node in facts['nodes']:
        if canonical[node['node_id']]!=node['node_id']:
            continue
        span=spans.get(node['source_span_id'],{})
        changed=any(c['file_path']==path and c['entity_name']==node['label']
                    and c['byte_start']==span.get('start_byte') and c['byte_end']==span.get('end_byte')
                    for c in semantic_changes)
        graph_nodes.append({'id':global_id(node['node_id']),'label':node['label'],'kind':node['kind'],
            'file_path':path,'changed':changed,'start_line':span.get('start_line'),
            'source_sha256':file_result['source_sha256'],
            'component':'Aggregate binding/rewrite' if 'first_last_any' in path else 'Relational statistics'})
    profile=Counter(e['relation'] for e in facts['edges'])
    resolved=0
    for edge in facts['edges']:
        if edge['relation']!='calls':
            continue
        item={'edge_id':RUN_ID+':'+path+':'+str(edge['edge_id']),
              'relation':edge['relation'],'confidence':edge['confidence'],
              'bind_method':edge['bind_method'],'file_path':path,'source_sha256':file_result['source_sha256'],
              'native_evidence_id':edge['evidence_id'],'evidence_kind':'native_single_file'}
        if edge['source_node_id'] in local and edge['target_node_id'] in local:
            item.update(src=global_id(edge['source_node_id']),dst=global_id(edge['target_node_id']))
            call_edges.append(item)
            resolved+=1
        else:
            item.update(source_node_id=edge['source_node_id'],target_node_id=edge['target_node_id'],
                        target_label=edge['target_label'])
            unresolved_calls.append(item)
    profiles.append({'file_path':path,'relations':dict(profile),'resolved_call_sites':resolved,
                     'unresolved_call_sites':sum(e['relation']=='calls' for e in facts['edges'])-resolved})
for name,data in [('graph-nodes',graph_nodes),('resolved-calls',call_edges),
                  ('unresolved-calls',unresolved_calls),('edge-profiles',profiles)]:
    save(name,data)
duckdb.execute("CREATE OR REPLACE TABLE native_nodes AS SELECT * FROM read_json_auto(?)",[str(ARTIFACT/'graph-nodes.json')])
duckdb.execute("CREATE OR REPLACE TABLE native_calls AS SELECT * FROM read_json_auto(?)",[str(ARTIFACT/'resolved-calls.json')])
duckdb.execute("CREATE OR REPLACE TABLE native_unresolved_calls AS SELECT * FROM read_json_auto(?)",[str(ARTIFACT/'unresolved-calls.json')])
assert len({n['id'] for n in graph_nodes})==len(graph_nodes)
assert len([n for n in graph_nodes if n['changed']])==5
assert all(e['src'] in {n['id'] for n in graph_nodes} and e['dst'] in {n['id'] for n in graph_nodes} for e in call_edges)
print('Native relation profiles:',profiles)
print('Graph table counts:',len(graph_nodes),'vertices,',len(call_edges),'resolved call sites,',len(unresolved_calls),'unresolved call sites')
print('Changed graph vertices:',rows("SELECT label,file_path,start_line FROM native_nodes WHERE changed ORDER BY label"))
print('Resolved call samples:',rows("SELECT s.label AS caller,t.label AS callee,e.confidence,e.bind_method FROM native_calls e JOIN native_nodes s ON s.id=e.src JOIN native_nodes t ON t.id=e.dst WHERE s.changed OR t.changed LIMIT 12"))


Native relation profiles: [{'file_path': 'src/function/aggregate/distributive/first_last_any.cpp', 'relations': {'contains': 82, 'defines': 81, 'calls': 139, 'extends': 4}, 'resolved_call_sites': 1, 'unresolved_call_sites': 138}, {'file_path': 'src/optimizer/relation_statistics/relation_statistics_helper.cpp', 'relations': {'contains': 34, 'defines': 33, 'imports': 1, 'calls': 227}, 'resolved_call_sites': 16, 'unresolved_call_sites': 211}]
Graph table counts: 94 vertices, 17 resolved call sites, 349 unresolved call sites
Changed graph vertices: [{'label': 'ExtractAggregationStatsInternal', 'file_path': 'src/optimizer/relation_statistics/relation_statistics_helper.cpp', 'start_line': 558}, {'label': 'GetValuePreservingColumn', 'file_path': 'src/optimizer/relation_statistics/relation_statistics_helper.cpp', 'start_line': 455}, {'label': 'HasBaseColumnSource', 'file_path': 'src/optimizer/relation_statistics/relation_statistics_helper.cpp', 'start_line': 476}, {'label': 'HasUniqueGroups', 

In [11]:
-- Real property graph from the native Spur Graph fact rows.
-- Calls are directed caller -> callee. Reverse impact asks which callers reach a changed callee.
ALTER TABLE native_nodes ADD PRIMARY KEY (id);
CREATE OR REPLACE PROPERTY GRAPH native_pr_graph
  VERTEX TABLES (native_nodes LABEL symbol)
  EDGE TABLES (native_calls
    SOURCE KEY (src) REFERENCES native_nodes (id)
    DESTINATION KEY (dst) REFERENCES native_nodes (id)
    LABEL calls);

CREATE OR REPLACE TABLE native_blast_radius AS
SELECT * FROM GRAPH_TABLE(native_pr_graph
  MATCH p = ANY SHORTEST (caller:symbol)-[e:calls]->{1,2}(changed:symbol)
  WHERE changed.changed = true AND caller.id <> changed.id
  COLUMNS (caller.id AS caller_id, caller.label AS caller_name,
           caller.start_line AS caller_line, caller.component AS component,
           changed.id AS changed_id,changed.label AS changed_name,
           path_length(p) AS hops))
ORDER BY changed_name,hops,caller_line;

SELECT caller_name,caller_line,changed_name,hops,component
FROM native_blast_radius ORDER BY changed_name,hops,caller_line;

caller_name,caller_line,changed_name,hops,component
RelationStatisticsHelper::ExtractAggregationStats,688,ExtractAggregationStatsInternal,1,Relational statistics
RelationStatisticsHelper::ExtractAggregationStats,696,ExtractAggregationStatsInternal,1,Relational statistics
HasBaseColumnSource,476,GetValuePreservingColumn,1,Relational statistics
HasUniqueGroups,513,GetValuePreservingColumn,1,Relational statistics
ExtractAggregationStatsInternal,558,GetValuePreservingColumn,1,Relational statistics
RelationStatisticsHelper::ExtractAggregationStats,688,GetValuePreservingColumn,2,Relational statistics
RelationStatisticsHelper::ExtractAggregationStats,696,GetValuePreservingColumn,2,Relational statistics
ExtractAggregationStatsInternal,558,HasBaseColumnSource,1,Relational statistics
RelationStatisticsHelper::ExtractAggregationStats,688,HasBaseColumnSource,2,Relational statistics
RelationStatisticsHelper::ExtractAggregationStats,696,HasBaseColumnSource,2,Relational statistics


## What the native graph proves

The native relation is **caller → callee**. The query asks which callers can reach a changed callee within one or two call edges. It uses actual resolved call facts and preserves distinct overload IDs and source lines.

These two C++ files produce **17 resolved call sites and 349 unresolved call sites** at the pinned head. The extension reports single-file resolution. Unresolved calls, callback registration and cross-file metadata flow cannot be recovered by graph traversal alone.

A blank result for a changed symbol means no path was found in this limited resolved graph. It does not establish zero impact. The 94 displayed vertices use an explicit containment normalization for overlapping function/template representations; raw native facts are retained.

The next experiments keep history and the previously source-inspected cross-file map in separate evidence layers. Their edges must not be presented as automatically resolved native calls.


In [12]:
# Base-pinned historical expertise; GitHub current participants stay in separate columns.
history,blame = [],[]
save('history-schema',rows('DESCRIBE SELECT * FROM git_log(?)',[REPO]))
save('blame-schema',rows('DESCRIBE SELECT * FROM git_blame(?,repo_path:=?,revision:=?)',
                         [FILES[0],REPO,MERGE_BASE]))
for path in FILES:
    uri=duckdb.execute('SELECT git_uri(?,?,?)',[REPO,path,MERGE_BASE]).fetchall()[0][0]
    records=rows("""SELECT commit_hash,author_name,author_date,commit_date,message FROM git_log(?)
                    WHERE author_date >= TIMESTAMP '2026-05-21'
                    ORDER BY author_date DESC LIMIT 40""",[uri])
    for r in records:
        r['file_path']=path
    history.extend(records)
    low,high=(19,60) if 'first_last_any' in path else (450,565)
    blame.extend(rows("""SELECT file_path,revision,line_number,commit_hash,author_name,author_date
                        FROM git_blame(?,repo_path:=?,revision:=?,min_line:=?,max_line:=?,
                                       ignore_whitespace:=true,use_mailmap:=true)
                        ORDER BY line_number""",[path,REPO,MERGE_BASE,low,high]))
save('base-history',history)
save('base-blame',blame)
duckdb.execute('CREATE OR REPLACE TABLE poc_history AS SELECT * FROM read_json_auto(?)',[str(ARTIFACT/'base-history.json')])
duckdb.execute('CREATE OR REPLACE TABLE poc_blame AS SELECT * FROM read_json_auto(?)',[str(ARTIFACT/'base-blame.json')])
# Reuse explicit identity evidence from the earlier source/commit-inspected experiment.
PRIOR=Path('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-17-duckdb-pr-28731')
identity_evidence=json.loads((PRIOR/'github-identity-evidence.json').read_text())
assert {r['github_author'] for r in identity_evidence}=={'lnkuiper','kryonix'}
save('github-identity-evidence',identity_evidence)
duckdb.execute("CREATE OR REPLACE TABLE identity_map(author_name VARCHAR,github_login VARCHAR,evidence_url VARCHAR)")
duckdb.executemany("INSERT INTO identity_map VALUES (?,?,?)",[
 ('Laurens Kuiper','lnkuiper',next(r['html_url'] for r in identity_evidence if r['github_author']=='lnkuiper')),
 ('Denis Hirn','kryonix',next(r['html_url'] for r in identity_evidence if r['github_author']=='kryonix'))])
duckdb.execute("""
 CREATE OR REPLACE TABLE stakeholder_candidates AS
 WITH b AS (SELECT file_path,author_name,count(*) AS base_region_lines FROM poc_blame GROUP BY ALL),
 h AS (SELECT file_path,author_name,count(*) AS history_commits FROM poc_history GROUP BY ALL)
 SELECT b.*,i.github_login,i.evidence_url,h.history_commits,
        'historical_expertise_candidate' AS role,false AS formal_owner_verified
 FROM b LEFT JOIN h USING(file_path,author_name) LEFT JOIN identity_map i USING(author_name)
""")
save('stakeholder-candidates',rows('SELECT * FROM stakeholder_candidates ORDER BY file_path,base_region_lines DESC'))
save('github-inbox',{'source':'attached @github-pat SQL functions','captured_at_utc':datetime.now(timezone.utc).isoformat(),
                    'pull_requests':rows('SELECT * FROM github_poc_pr_inbox'),
                    'issues':rows('SELECT * FROM github_poc_issue_inbox'),
                    'scope':'two targeted searches, not a complete repository inbox'})
print('Historical rows:',len(history),'blame rows:',len(blame))
print('Strongest mapped candidates:',rows('SELECT author_name,github_login,base_region_lines,history_commits FROM stakeholder_candidates WHERE github_login IS NOT NULL ORDER BY author_name'))


Historical rows: 65 blame rows: 158
Strongest mapped candidates: [{'author_name': 'Denis Hirn', 'github_login': 'kryonix', 'base_region_lines': 38, 'history_commits': 7}, {'author_name': 'Laurens Kuiper', 'github_login': 'lnkuiper', 'base_region_lines': 108, 'history_commits': 22}]


In [13]:
# A separate graph layer uses previously source-inspected cross-file relationships.
# This demonstrates query composition, not automated repository linking.
curated_nodes=json.loads((PRIOR/'source-evidence.json').read_text())
curated_edges=[e for e in json.loads((PRIOR/'impact-edges.json').read_text()) if e['evidence_class']=='source_path']
for n in curated_nodes:
    assert n['revision']==HEAD
    assert hashlib.sha256(git('show',HEAD+':'+n['path'])).hexdigest()==n['source_sha256']
    n['evidence_origin']='prior_source_inspection_revalidated_blob_hash'
save('curated-nodes',curated_nodes)
save('curated-edges',curated_edges)
duckdb.execute('CREATE OR REPLACE TABLE curated_nodes AS SELECT * FROM read_json_auto(?)',[str(ARTIFACT/'curated-nodes.json')])
duckdb.execute('CREATE OR REPLACE TABLE curated_edges AS SELECT * FROM read_json_auto(?)',[str(ARTIFACT/'curated-edges.json')])
duckdb.execute('ALTER TABLE curated_nodes ADD PRIMARY KEY(id)')
# SQL recursion independently checks the DuckPGQ result on the actual native call graph.
sql_radius=rows("""
 WITH RECURSIVE r(caller_id,changed_id,hops) AS (
   SELECT src,dst,1 FROM native_calls
   UNION ALL
   SELECT r.caller_id,e.dst,r.hops+1 FROM r JOIN native_calls e ON r.changed_id=e.src WHERE r.hops<2
 )
 SELECT r.caller_id,r.changed_id,min(r.hops) AS hops
 FROM r JOIN native_nodes t ON t.id=r.changed_id
 WHERE t.changed AND r.caller_id<>r.changed_id GROUP BY r.caller_id,r.changed_id
""")
pgq_radius=rows('SELECT caller_id,changed_id,hops FROM native_blast_radius')
assert sorted(tuple(r.values()) for r in sql_radius)==sorted(tuple(r.values()) for r in pgq_radius)
save('native-blast-radius',rows('SELECT * FROM native_blast_radius ORDER BY changed_name,hops,caller_id'))
save('graph-parity',{'duckpgq_rows':len(pgq_radius),'recursive_sql_rows':len(sql_radius),'identical':True,
                     'direction':'caller -> changed callee','max_hops':2,'scope':'two changed C++ files, resolved native calls only'})
print('DuckPGQ/native graph parity:',len(pgq_radius),'rows match recursive SQL exactly.')
print('Curated cross-file layer:',len(curated_nodes),'source-inspected points;',len(curated_edges),'structural relationships')
print('No architecture contract is approved for this POC; alignment remains not evaluated.')


DuckPGQ/native graph parity: 13 rows match recursive SQL exactly.
Curated cross-file layer: 20 source-inspected points; 14 structural relationships
No architecture contract is approved for this POC; alignment remains not evaluated.


In [14]:
-- Separate curated propagation graph, preserving its evidence origin.
CREATE OR REPLACE PROPERTY GRAPH curated_pr_graph
  VERTEX TABLES (curated_nodes LABEL source_point)
  EDGE TABLES (curated_edges
    SOURCE KEY (from_node) REFERENCES curated_nodes (id)
    DESTINATION KEY (to_node) REFERENCES curated_nodes (id)
    LABEL propagates);
CREATE OR REPLACE TABLE curated_propagation AS
SELECT * FROM GRAPH_TABLE(curated_pr_graph
  MATCH p = ANY SHORTEST (a:source_point)-[e:propagates]->{1,6}(b:source_point)
  WHERE a.id='ordered' AND b.id IN ('manager','estimator','cost')
  COLUMNS (a.statement AS changed_behavior,b.id AS downstream_id,
           b.component AS downstream_component,b.statement AS downstream_behavior,
           b.url AS source_url,path_length(p) AS hops))
ORDER BY hops;
SELECT downstream_id,downstream_component,hops,source_url FROM curated_propagation ORDER BY hops;

downstream_id,downstream_component,hops,source_url
manager,Join optimization,2,https://github.com/duckdb/duckdb/blob/d29b26bb92ac940a0807be0f43cc385b31623839/src/optimizer/join_order/relation_manager.cpp#L623
estimator,Join optimization,4,https://github.com/duckdb/duckdb/blob/d29b26bb92ac940a0807be0f43cc385b31623839/src/optimizer/join_order/cardinality_estimator.cpp#L987
cost,Join optimization,5,https://github.com/duckdb/duckdb/blob/d29b26bb92ac940a0807be0f43cc385b31623839/src/optimizer/join_order/plan_enumerator.cpp#L200


In [15]:
-- Component membership is an explicit POC path rule, not an approved architecture policy.
CREATE OR REPLACE TABLE component_rollup AS
SELECT n.component,count(*) AS normalized_vertices,
       count(*) FILTER (WHERE n.changed) AS directly_changed_symbols,
       count(DISTINCT b.caller_id) AS caller_vertices_in_limited_graph,
       'POC path mapping' AS mapping_source,
       'not_evaluated_no_architecture_policy' AS alignment_state
FROM native_nodes n LEFT JOIN
 (SELECT DISTINCT caller_id FROM native_blast_radius) b ON b.caller_id=n.id
GROUP BY n.component;
SELECT component,directly_changed_symbols,caller_vertices_in_limited_graph,alignment_state
FROM component_rollup ORDER BY component;

component,directly_changed_symbols,caller_vertices_in_limited_graph,alignment_state
Aggregate binding/rewrite,1,0,not_evaluated_no_architecture_policy
Relational statistics,4,5,not_evaluated_no_architecture_policy


In [16]:
-- Provenance-aware people view: current participants and historical candidates have separate roles.
SELECT user_login AS person,'PR author (live inbox)' AS role,
       title AS evidence,NULL::BIGINT AS base_region_lines
FROM github_poc_pr_inbox
UNION ALL
SELECT user_login,'issue reporter (live inbox)',title,NULL::BIGINT FROM github_poc_issue_inbox
UNION ALL
SELECT github_login,'historical expertise candidate',file_path,base_region_lines
FROM stakeholder_candidates WHERE github_login IS NOT NULL
ORDER BY role,person;

person,role,evidence,base_region_lines
kevintruong,PR author (live inbox),Preserve aggregate domains for proven singleton groups,NULL
kryonix,historical expertise candidate,src/function/aggregate/distributive/first_last_any.cpp,38
lnkuiper,historical expertise candidate,src/optimizer/relation_statistics/relation_statistics_helper.cpp,108
ted-wq-x,issue reporter (live inbox),Performance regression after #28456: DISTINCT ON loses join-key statistics and changes join order,NULL


In [17]:
# Audit local IDs/normalization and the DuckDB 2.0 Python-to-Arrow seam for a future app.
aliases=[]
for result in after_facts:
    facts=result['facts']
    spans={s['span_id']:s for s in facts['spans']}
    for n in facts['nodes']:
        s=spans.get(n['source_span_id'])
        if not s or n['kind'] not in ('function','method'):
            continue
        containers=[o for o in facts['nodes'] if o['kind']==n['kind'] and o['label']==n['label']
                    and o['source_span_id'] in spans
                    and spans[o['source_span_id']]['start_byte']<=s['start_byte']
                    and spans[o['source_span_id']]['end_byte']>=s['end_byte']
                    and spans[o['source_span_id']]['end_byte']-spans[o['source_span_id']]['start_byte']>s['end_byte']-s['start_byte']]
        if containers:
            canonical=min(containers,key=lambda o:o['node_id'])
            aliases.append({'file_path':result['file_path'],'raw_node_id':n['node_id'],
                            'canonical_node_id':canonical['node_id'],'label':n['label'],
                            'reason':'same-name same-kind containing source span; experimental template normalization'})
save('node-aliases',aliases)
assert len(aliases)==sum(len(r['facts']['nodes']) for r in after_facts)-len(graph_nodes)
relation=duckdb.sql('SELECT * FROM component_rollup ORDER BY component')
arrow_api={name:hasattr(relation,name) for name in ['fetch_arrow_table','to_arrow_table','arrow']}
arrow_table=relation.to_arrow_table()
assert arrow_table.num_rows==2
save('arrow-seam',{'duckdb':duckdb.__version__,'relation_methods':arrow_api,
                    'actual_method':'to_arrow_table','rows':arrow_table.num_rows,
                    'columns':arrow_table.column_names,'spur_injected': 'spur' in globals()})
print('Normalization aliases:',len(aliases),'; native numeric IDs are per-file and require scoped keys.')
print('DuckDB relation Arrow API:',arrow_api)
print('Actual Arrow conversion succeeded:',arrow_table.num_rows,'component rows')


Normalization aliases: 24 ; native numeric IDs are per-file and require scoped keys.
DuckDB relation Arrow API: {'fetch_arrow_table': True, 'to_arrow_table': True, 'arrow': True}
Actual Arrow conversion succeeded: 2 component rows


In [18]:
-- End-to-end join: @github-pat inbox + Spur Graph changes + DuckPGQ paths + DuckTails expertise.
CREATE OR REPLACE TABLE review_poc_join AS
SELECT p.number AS pr_number,n.label AS changed_symbol,n.component,
       count(DISTINCT b.caller_id) AS callers_found_in_limited_graph,
       s.github_login AS historical_expertise_candidate,s.base_region_lines,
       'partial_single_file' AS impact_coverage,
       'not_evaluated_no_policy' AS architecture_alignment
FROM github_poc_pr_inbox p
JOIN native_nodes n ON n.changed
LEFT JOIN native_blast_radius b ON b.changed_id=n.id
LEFT JOIN stakeholder_candidates s ON s.file_path=n.file_path AND s.github_login IS NOT NULL
WHERE p.number=28731
GROUP BY p.number,n.id,n.label,n.component,s.github_login,s.base_region_lines;
SELECT changed_symbol,callers_found_in_limited_graph,historical_expertise_candidate,
       impact_coverage,architecture_alignment
FROM review_poc_join ORDER BY changed_symbol;

changed_symbol,callers_found_in_limited_graph,historical_expertise_candidate,impact_coverage,architecture_alignment
ExtractAggregationStatsInternal,2,lnkuiper,partial_single_file,not_evaluated_no_policy
GetValuePreservingColumn,5,lnkuiper,partial_single_file,not_evaluated_no_policy
HasBaseColumnSource,3,lnkuiper,partial_single_file,not_evaluated_no_policy
HasUniqueGroups,3,lnkuiper,partial_single_file,not_evaluated_no_policy
RewriteOrderedFirst,0,kryonix,partial_single_file,not_evaluated_no_policy


In [19]:
# Export one evidence packet an app or reviewing agent can consume.
# This creates a packet; it does not execute an AI model.
legacy_arrow_probe={}
try:
    legacy_table=duckdb.sql('SELECT 1 AS probe').fetch_arrow_table()
    legacy_arrow_probe={'status':'succeeded','rows':legacy_table.num_rows}
except Exception as error:
    legacy_arrow_probe={'status':'failed','error_type':type(error).__name__,'message':str(error)[:300]}
save('legacy-arrow-probe',legacy_arrow_probe)
inbox=json.loads((ARTIFACT/'github-inbox.json').read_text())
packet={
 'schema_version':1,'source_snapshot_id':RUN_ID,'repository':'duckdb/duckdb','pr_number':28731,
 'comparison':json.loads((ARTIFACT/'comparison.json').read_text()),
 'github':{'inbox':inbox,'metadata_source':'GitHub connector get_pr_info',
           'pr_context_captured_at_utc':PR['captured_at_utc'],
           'requested_reviewers_state':'unknown' if PR['requested_reviewers'] is None else 'known'},
 'changes':{'semantic_function_count':len(semantic_changes),'raw_candidate_count':len(raw_candidates),
            'items':[dict((k,c[k]) for k in ['file_path','entity_name','change','source_sha256','byte_start','byte_end','line_start','line_end']) for c in semantic_changes]},
 'impact':{'native_scope':'two changed C++ files; resolved call graph; caller to callee',
           'normalized_vertices':len(graph_nodes),'resolved_call_sites':len(call_edges),
           'unresolved_call_sites':len(unresolved_calls),'max_hops':2,
           'caller_seed_rows':rows('SELECT * FROM native_blast_radius ORDER BY changed_name,hops,caller_id'),
           'coverage':'partial_single_file','whole_repository_blast_radius_established':False},
 'curated_propagation':{'origin':'source-inspected prior map; source hashes revalidated',
                       'automatically_native_resolved':False,'items':rows('SELECT * FROM curated_propagation ORDER BY hops')},
 'architecture':{'mapping_source':'explicit POC path rules and source-inspected labels',
                 'approved_policy':None,'alignment_state':'not_evaluated',
                 'components':rows('SELECT * FROM component_rollup ORDER BY component')},
 'people':{'history_base':MERGE_BASE,'history_since':'2026-05-21','max_commits_per_file':40,
           'blame_regions':{'first_last_any.cpp':[19,60],'relation_statistics_helper.cpp':[450,565]},
           'candidates':rows('SELECT * FROM stakeholder_candidates ORDER BY file_path,base_region_lines DESC'),
           'formal_ownership_verified':False,'current_assignment_state':'unknown'},
 'source_coverage':{'analyzed_files':FILES,
                    'other_changed_files_not_symbol_analyzed':[r['file_path'] for r in pr_files if r['file_path'] not in FILES]},
 'joined_review_rows':rows('SELECT * FROM review_poc_join ORDER BY changed_symbol'),
 'ai_review':{'status':'not_run','findings':[]},
 'validation':{'native_pgq_matches_recursive_sql':True,'runtime_benchmarks_run':False,'pr_regression_suite_run':False},
 'known_gaps':['repository linking and callback/metadata edges',
               'ordinary symbol matching and experimental normalization',
               'GitHub detailed-resource collection','approved architecture policy',
               'current ownership/assignment evidence','AI provider execution']
}
packet['packet_id']=hashlib.sha256(json.dumps(packet,sort_keys=True,default=str).encode()).hexdigest()
save('review-evidence-packet',packet)
for name,sql in [('native-blast-radius','SELECT * FROM native_blast_radius ORDER BY changed_name,hops,caller_id'),
                 ('curated-propagation','SELECT * FROM curated_propagation ORDER BY hops'),
                 ('component-rollup','SELECT * FROM component_rollup ORDER BY component'),
                 ('review-join','SELECT * FROM review_poc_join ORDER BY changed_symbol')]:
    save(name,rows(sql))
binary_inventory=[{'path':str(p),'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size}
                  for p in sorted(EXTENSIONS.rglob('*.duckdb_extension'))]
save('extension-binaries',binary_inventory)
print('Packet:',packet['packet_id'],'with',len(packet['joined_review_rows']),'joined review rows')
print('Current legacy Arrow alias probe:',legacy_arrow_probe)
print('Installed binaries:',[(Path(b['path']).name,b['bytes']) for b in binary_inventory])
print('AI review status:',packet['ai_review']['status'])


Packet: 6be0513b109854a1ab64bcaf9b0467320c6ee4c2cae29a89b3f28756848eb3dd with 5 joined review rows
Current legacy Arrow alias probe: {'status': 'succeeded', 'rows': 1}
Installed binaries: [('duck_tails.duckdb_extension', 12163438), ('duckpgq.duckdb_extension', 40862174), ('spur_graph.duckdb_extension', 17417686)]
AI review status: not_run


/var/folders/mv/vtzfmys92yl9rwcsgf9n1x780000gn/T/ipykernel_87570/2692208317.py:5: DeprecationWarning: fetch_arrow_table() is deprecated, use to_arrow_table() instead.
  legacy_table=duckdb.sql('SELECT 1 AS probe').fetch_arrow_table()


## POC result and remaining capability gaps

**Proven in one DuckDB 2.0 connection:** three extensions install/load together; the saved @github-pat connection returns real PR/issue inbox rows; pinned native extraction finds five semantic change seeds; DuckPGQ traverses native call facts; the result matches recursive SQL; DuckTails supplies base-history/blame; SQL joins these into five review rows and exports an evidence packet.

**Measured scope:** 94 normalized vertices, 17 resolved call sites, 349 unresolved call sites and 13 caller/changed-symbol relationships within two hops. The broader source-inspected map is a separate curated graph, not automatic cross-file resolution. No whole-repository blast radius is established.

**Still needed before the app:** repository-wide linking and callback/data-flow evidence; robust production change normalization; richer GitHub detail/review/check collection; an approved architecture policy; verified current ownership/assignment; configured AI review execution.

**Python/Arrow finding:** to_arrow_table() works. The deprecated fetch_arrow_table() alias also works in this kernel, so the prior app failure cannot be generalized as an absent method in every DuckDB 2.0 environment. The actual app transport needs its own environment-specific regression check.

**Operational observations:** the first two-resource SQL batch returned a failure without diagnostics; the unchanged retry succeeded. One subsequent notebook bridge mutation timed out and succeeded via the same idempotency key. These observations are preserved, with no unsupported root-cause claim.

The exported packet keeps architecture alignment as not evaluated, current reviewer assignment as unknown and AI review as not run. These statuses are evidence, not empty successful results.


In [20]:
# Verification of this POC's claims. No upstream PR tests or runtime benchmark is implied.
checks={}
inventory=rows("SELECT extension_name,loaded FROM duckdb_extensions() WHERE extension_name IN ('duck_tails','spur_graph','duckpgq')")
assert len(inventory)==3 and all(r['loaded'] for r in inventory)
checks['three_extensions_loaded']=True
assert rows("SELECT number FROM github_poc_pr_inbox")==[{'number':28731}]
assert rows("SELECT number FROM github_poc_issue_inbox")==[{'number':28615}]
checks['saved_github_connection_real_inbox']=True
assert len(pr_files)==5 and len(semantic_changes)==5 and len(raw_candidates)==6
assert all(i['status']=='ok' for i in native_inputs)
for i in native_inputs:
    assert hashlib.sha256(git('show',i['revision']+':'+i['file_path'])).hexdigest()==i['source_sha256']
checks['pinned_source_and_native_extraction']=True
assert len(graph_nodes)==94 and len(call_edges)==17 and len(unresolved_calls)==349
assert len(aliases)==24
assert rows("SELECT count(*) AS n FROM native_nodes WHERE changed")==[{'n':5}]
assert rows("SELECT count(*) AS n FROM native_blast_radius")==[{'n':13}]
assert json.loads((ARTIFACT/'graph-parity.json').read_text())['identical']
checks['native_property_graph_and_sql_parity']=True
assert rows("SELECT downstream_id,hops FROM curated_propagation ORDER BY hops")==[
 {'downstream_id':'manager','hops':2},{'downstream_id':'estimator','hops':4},{'downstream_id':'cost','hops':5}]
checks['curated_layer_separate_and_queryable']=True
assert len(history)==65 and len(blame)==158
for commit in {r['commit_hash'] for r in history}:
    subprocess.run(['git','-C',REPO,'merge-base','--is-ancestor',commit,MERGE_BASE],check=True,timeout=10)
checks['base_pinned_history_and_blame']=True
assert len(packet['joined_review_rows'])==5
assert packet['architecture']['alignment_state']=='not_evaluated'
assert packet['people']['current_assignment_state']=='unknown'
assert packet['ai_review']['status']=='not_run'
checks['joined_packet_preserves_evidence_limits']=True
assert arrow_table.num_rows==2 and legacy_arrow_probe['status']=='succeeded'
checks['actual_python_arrow_conversion']=True
for record in binary_inventory:
    assert hashlib.sha256(Path(record['path']).read_bytes()).hexdigest()==record['sha256']
checks['installed_binary_hashes']=True
receipt={'captured_at_utc':datetime.now(timezone.utc).isoformat(),'runtime':duckdb.__version__,
         'checks':checks,'source_snapshot_id':RUN_ID,'packet_id':packet['packet_id'],
         'counts':{'changed_files':len(pr_files),'semantic_functions':len(semantic_changes),
                   'normalized_vertices':len(graph_nodes),'resolved_call_sites':len(call_edges),
                   'unresolved_call_sites':len(unresolved_calls),'native_caller_seed_pairs':len(pgq_radius),
                   'history_rows':len(history),'blame_rows':len(blame),'joined_review_rows':5},
         'scope':'experimental notebook data plane; no app frontend, AI execution or PR correctness certification'}
save('verification',receipt)
print(json.dumps(receipt,indent=2))


{
  "captured_at_utc": "2026-11-17T16:01:58.417646+00:00",
  "runtime": "2.0.0",
  "checks": {
    "three_extensions_loaded": true,
    "saved_github_connection_real_inbox": true,
    "pinned_source_and_native_extraction": true,
    "native_property_graph_and_sql_parity": true,
    "curated_layer_separate_and_queryable": true,
    "base_pinned_history_and_blame": true,
    "joined_packet_preserves_evidence_limits": true,
    "actual_python_arrow_conversion": true,
    "installed_binary_hashes": true
  },
  "source_snapshot_id": "cc714dc606372787dc8b",
  "packet_id": "6be0513b109854a1ab64bcaf9b0467320c6ee4c2cae29a89b3f28756848eb3dd",
  "counts": {
    "changed_files": 5,
    "semantic_functions": 5,
    "normalized_vertices": 94,
    "resolved_call_sites": 17,
    "unresolved_call_sites": 349,
    "native_caller_seed_pairs": 13,
    "history_rows": 65,
    "blame_rows": 158,
    "joined_review_rows": 5
  },
  "scope": "experimental notebook data plane; no app frontend, AI execution or P